# 6e · E2E relation canonicalization  (twin of nb7b, for the single-call graph)

Runs the **same** predicate canonicalization as nb7b, on the **e2e** graph. For each distinct
free-form `relation_phrase` it retrieves candidate properties from **schema.org ∪ e2e property
vocabulary**, and ONE LLM call per article assigns every relation instance a property — reusing a
candidate `as_is`, reusing one in the opposite direction (`inverse`, and the stored triple is
swapped to match), or minting a new `ex:` property. A phrase may **split** across several properties
when its instances have different type signatures. Instances are then deduplicated deterministically
on `(subject, property, object)`.

- **Config:** one line — `EXPERIMENT`; everything else comes from `configs/<EXPERIMENT>.json`, the
  same file 6c and 6d read, so `INPUT_TAG` here always names the 6d run this chain produced.
- **Input:** `nary_graph<_isfx>_typed_consolidated.json` — **6d's consolidated output**, so 6d's
  re-consolidation block must have been run.
- **Output:** `nary_graph<_osfx>_extended.json` (the final e2e KG) + `nary_relation_matching_review<_osfx>.json`,
  and, after the re-consolidation block at the bottom, `nary_graph<_osfx>_extended_consolidated.json`.
- **Separate vocabulary:** grows `_global/e2e/nary_relation_vocabulary<_osfx>.json`, independent of
  the staged track's, so the two methods' predicate economies stay comparable.

In [1]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [2]:
# ---- WHICH EXPERIMENT ---------------------------------------------------------------------
# ONE name decides the whole run. Every knob below comes from configs/<EXPERIMENT>.json — the SAME
# file 6c and 6d read. That is what removes the failure this notebook used to be most exposed to:
# INPUT_TAG naming a 6d run the chain never produced.
#   list what exists :  python -m experiment_config list          (run from notebooks/)
#   see one resolved :  python -m experiment_config show main
# $PHASE2_EXPERIMENT wins when it is set, so a headless batch of ablations (run_e2e.sh) can sweep
# experiments without editing a single notebook.
import os
EXPERIMENT = os.environ.get("PHASE2_EXPERIMENT") or "main"

import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "experiment_config.py").exists():
        sys.path.insert(0, str(_p)); break
import experiment_config as xcfg; importlib.reload(xcfg)

# Sets the shared keys (RESULTS_MODEL, BACKEND, GEMINI_MODEL, MATCH_MODEL, VARIANT, RUN_TAG,
# INPUT_TAG, SELECTED_ARTICLES) and this section's own:
#   FRESH / EXPAND_MODE / TOP_K_MATCH / ALIGN_QUERY_STYLE
#   INLINE_VOCAB_CANDIDATES                   which pool the per-article selection call sees
#   PARALLEL_LLM / WORKERS                    overlap the calls when they are independent
#   ANCHOR_MODE / ANCHOR_PENALTY / DROP_AMBIGUOUS_SYNONYMS      how a property is indexed
#   RERANK / RERANK_FROM / RERANK_MODEL       inline cross-encoder stage
#   CONSOLIDATE / CONSOLIDATE_TOP_K / CONSOLIDATE_INCLUDE_SCHEMA / SUBPROP_MODE
#   ALLOW_GENERALIZE / TWO_CALL_CONSOLIDATION / GUARD_SCHEMA_PARENT
#   RERANK_CONSOLIDATE / RERANK_CONSOLIDATE_FROM                consolidation cross-encoder stage
#   INCLUDE_ALL_SCHEMA                        how much of schema.org the printed hierarchy shows
CFG = xcfg.load(EXPERIMENT, globals(), "6e")
# A one-off try goes RIGHT HERE, below the load, where it wins over the file.

# --- imports + derived paths ---------------------------------------------------------------
import json, re, os, copy, time
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

SCHEMA_DIR  = ROOT / "schema_org_files"          # schema.org PROPERTY pool from nb5
RESULTS_DIR = ROOT / "results" / RESULTS_MODEL

# THREE suffixes, because this notebook consumes one run and produces another:
#   _sfx   = "_" + VARIANT              the 6c extraction arm both stages work on
#   _isfx  = _sfx + "_" + INPUT_TAG     which 6d run to READ  (6d's own RUN_TAG)
#   _osfx  = _sfx + "_" + RUN_TAG       everything THIS notebook WRITES
# INPUT_TAG defaults to RUN_TAG in the config, which is right when 6d and 6e run as one chain.
# Differ them (in the config file, not here) to vary something in 6e ALONE — a prompt, a rerank
# setting — on top of an existing 6d output.
_sfx  = f"_{VARIANT}" if VARIANT else ""
_isfx = _sfx + (f"_{INPUT_TAG}" if INPUT_TAG else "")
_osfx = _sfx + (f"_{RUN_TAG}" if RUN_TAG else "")

VOCAB_PATH = RESULTS_DIR / f"_global/e2e/nary_relation_vocabulary{_osfx}.json"   # SEPARATE per run

# ---- RETRIEVAL / RERANK / PARALLELISM — what the config's knobs mean here -------------------
# INLINE_VOCAB_CANDIDATES = False: the TOP_K_MATCH candidates shown to the selection LLM are ALWAYS
# schema.org — the grown vocabulary is never offered inline, so the LLM cannot reuse an earlier mint
# and re-mints synonyms of terms it already created. Collapsing those is deferred to the GLOBAL
# re-consolidation at the bottom, which is therefore NOT optional under this setting. Judge the
# configuration on the POST-CONSOLIDATION vocabulary size, not on the inline mint rate.
#
# PARALLEL_LLM exploits exactly that: with the inline pool fixed to schema.org, every article is
# offered the SAME candidates and the calls are independent, so they can overlap. Applying the
# results still happens strictly in ARTICLE_ORDER, so the vocabulary grows exactly as it would
# sequentially and the written output is unchanged — only the API waiting is overlapped. It is
# ignored (with a warning) when the inline pool includes the vocabulary, because then article k's
# prompt genuinely depends on articles 1..k-1.
#
# RERANK and RERANK_CONSOLIDATE are two INDEPENDENT cross-encoder stages, each with its own depth;
# the depth is that stage's ceiling, since the cross-encoder can only promote what e5 fetched.

# Which articles this runs on — and the ORDER the vocabulary grows along (incremental: what article
# k mints depends on what k-1 added).
_available = sorted((p.relative_to(RESULTS_DIR).parts[0]
                     for p in RESULTS_DIR.glob(f"*/e2e/nary_graph{_isfx}_typed_consolidated.json")),
                    key=xcfg.article_key)
if SELECTED_ARTICLES is None:
    ARTICLE_ORDER = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    if missing:
        print("WARNING: skipping (no e2e/nary_graph_typed_consolidated.json):", missing)
    ARTICLE_ORDER = [a for a in SELECTED_ARTICLES if a in _available]
assert ARTICLE_ORDER, (
    f"No runnable articles. Looked for */e2e/nary_graph{_isfx}_typed_consolidated.json — 6d's "
    f"output for INPUT_TAG={INPUT_TAG!r} under experiment {EXPERIMENT!r}. Run 6d (including its "
    f"re-consolidation block) under the same experiment first.")
print(f"\nreads 6d run {INPUT_TAG!r} -> nary_graph{_isfx}_typed_consolidated.json")
print(f"writes run   {RUN_TAG!r} -> *{_osfx}*   |  vocab: {VOCAB_PATH.name}")
print(f"{len(ARTICLE_ORDER)} article(s), in this order: {ARTICLE_ORDER}")

experiment 'deepseek-loop-drop'  ·  section 6e (e2e · relation canonicalization)
--------------------------------------------------------------------------------
  TEMPORARY: finish the deepseek-v4-flash main_test e2e run over the 75 articles whose 6c extraction is sound. Five of the 80 hit the same thinking-mode repetition loop qwen-loop-fix documents, emitting the same Event hub hundreds of times: art19 (6c produced nothing: repetition loop filled the 65,536-token output ceiling); art18 (6c emitted 801 Event hubs from 2 distinct labels (400x duplication)); art28 (6c emitted 788 Event hubs from 21 distinct labels and ZERO relations; this is what killed 6e); art42 (6c emitted 110 Event hubs from 35 distinct labels (3.1x duplication)); art50 (6c emitted 709 Event hubs from 25 distinct labels (28x duplication)). 6e is re-run FRESH rather than resumed, because art18 was already applied in chunk 2 of the crashed run and contributed 11 minted properties to the vocabulary; resuming would kee

In [3]:
# First run only (sentence-transformers pulls torch; google-genai for the LLM stages).
# %pip install -q sentence-transformers google-genai numpy json-repair

In [4]:
# Embedding model (loaded once). e5 needs "query: " on the query side and "passage: " on documents;
# both the relation-phrase queries and every document (schema.org OR vocab property) go through the
# same convention — identical to nb4.
from sentence_transformers import SentenceTransformer
_MODEL = SentenceTransformer(MATCH_MODEL)

def _prefixes(name):
    n = name.lower()
    if "e5" in n:  return "query: ", "passage: "
    if "bge" in n: return "Represent this sentence for searching relevant passages: ", ""
    return "", ""
QP, DP = _prefixes(MATCH_MODEL)

def embed(texts, prefix):
    return np.asarray(_MODEL.encode([prefix + t for t in texts],
                                    normalize_embeddings=True, batch_size=64))
print("embedder ready:", MATCH_MODEL, "| query_prefix", repr(QP), "| doc_prefix", repr(DP))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedder ready: intfloat/e5-base-v2 | query_prefix 'query: ' | doc_prefix 'passage: '


In [5]:
# Schema.org PROPERTY document pool (fixed) — embedded once (from nb5). Vocabulary properties are
# embedded per article inside retrieval, since the vocabulary grows during the run.
schema_pool   = json.loads((SCHEMA_DIR / "schema_properties_enriched.json").read_text())
SCHEMA_LABELS = {p["label"] for p in schema_pool}
schema_emb    = embed([p["embedding_text"] for p in schema_pool], DP)
print(len(schema_pool), "schema.org properties embedded")

1521 schema.org properties embedded


In [6]:
# ---- ANCHOR_MODE / ANCHOR_PENALTY / DROP_AMBIGUOUS_SYNONYMS — set in the experiment config ----
# ANCHOR_MODE — how each property is represented in the document pool:
#   "original" : ONE vector per property   "about: A subject matter relation … Also called concerning, …"
#   "A"        : + bare surface anchors    'about' · 'concerning' · 'regarding'
#   "B"        : + surface + sense tag     'about: subject matter relation' · 'concerning: subject matter relation'
#   "C"        : SURFACE FORMS ONLY, on BOTH sides — no definitions anywhere. The query becomes the
#                bare relation phrase ('caused by'), the documents the bare label and synonyms
#                ('cause', 'causal relation'). 192 of 213 query definitions open with the word
#                "Indicates" and average 9.6 words, so most of a query's mass is boilerplate shared
#                with every other query — a better explanation of the flat 0.83 score band than
#                anything on the document side. C removes it from both sides. camelCase is split so
#                'causedBy' can meet the synonym 'caused by' (128 of 193 phrases are camelCase).
#                C keeps NO blended vector: it is a clean test, not a safety net.
# A property's score is the MAX over its anchors, so the rest of the pipeline is unchanged — it
# still sees one score per property. "original" reproduces the single-vector behaviour exactly.
#
# ANCHOR_PENALTY — a tiebreak SUBTRACTED from synonym anchors, so a synonym must win clearly to
# outrank a property's own label. It must be ADDITIVE: e5 cosine similarities are squeezed into
# ~0.75-0.96, so a multiplicative 0.9 would demand a raw similarity of ~0.94 — the top of the whole
# distribution — and would silently disqualify every synonym anchor. The typical rank1-rank2 gap is
# ~0.010, so 0.005 is half a place. 0.0 disables the penalty.
#
# DROP_AMBIGUOUS_SYNONYMS — skip synonyms claimed by more than one property ('issued by',
# 'belongs to'); see _usable_synonyms below for the second failure mode it also covers.

# --- Multi-anchor document pool (retrieval variants A / B) -------------------------------------
# Today every property is ONE vector, mean-pooled over "label: definition. Also called syn1, syn2…".
# Mean pooling dilutes the surface forms — the synonyms are in the text but contribute ~1/35 of the
# average — which is why candidates arrive in a flat score band (0.791-0.817 in art1) and the right
# property can sit at rank 7. A/B index each property SEVERAL times, once per surface form, and score
# it by the MAX over its anchors, so an OIE phrase only has to be close to ONE of them:
#
#   original : the blended vector, exactly as today
#   A        : blended + bare surface forms          'about' · 'concerning' · 'on the topic of'
#   B        : blended + surface + short sense tag   'about: subject matter relation' · …
#   C        : surface forms ONLY — 'about' · 'concerning' — and the QUERY drops its definition too
#
# A spreads the anchors furthest but loses the sense ('record' would pull weather queries towards
# `album`); B keeps a ~1:1 surface-to-context ratio, enough to pin the sense without swamping the
# surface form. Every mode KEEPS the blended vector, so the max can only match or beat today.
# Synonym anchors are docked ANCHOR_PENALTY (ADDITIVE — see the config cell for why not a factor).
def _surface(s):
    """camelCase/PascalCase -> spaced lowercase, so the phrase 'causedBy' can meet the synonym
    'caused by'. 128 of the 193 OIE phrases are camelCase, as is every schema.org label; the
    synonyms are already natural language and pass through unchanged."""
    s = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", " ", s or "")
    s = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", " ", s)          # HTTPServer -> HTTP Server
    return re.sub(r"[^0-9a-zA-Z]+", " ", s).strip().lower()

_SYN_RE  = re.compile(r"Also called ([^.]+)\.")
_EXAM_RE = re.compile(r"\s*Examples?:.*$", re.S)

def _gloss(text):
    """The definition itself — 'Also called …' and 'Examples: …' stripped off."""
    return _EXAM_RE.sub("", _SYN_RE.sub("", text or "")).strip()

def _synonyms(text):
    m = _SYN_RE.search(text or "")
    return [s.strip().strip('"').strip() for s in m.group(1).split(",") if s.strip()] if m else []

def _sense_tag(gloss, label):
    """Variant B's short sense hint. nb5's enrichment prompt mandates 'A <X> relation between …',
    so the head before ' between ' is a 2-4 word gloss of the relation TYPE (holds for ~64% of
    properties); otherwise fall back to the first clause, then to the label."""
    if not gloss:
        return label
    head = gloss.split(" between ")[0] if " between " in gloss else re.split(r"[,.;]", gloss)[0]
    head = re.sub(r"^(an?|the)\s+", "", head.strip(), flags=re.I).strip()
    return head[:60] or label

# A synonym anchor is only useful if it points at ONE property. Two ways it fails to:
#   * claimed by several properties  — 'issued by' belongs to BOTH publisher and recordLabel,
#     'belongs to' to ten. As an anchor it drags every query towards all its owners (6g's failure).
#   * it IS another property's label — 'value' is listed as a synonym of `amount`, but schema.org
#     has an actual property called `value`, which should own that anchor outright (35 such cases).
# Both are dropped; a property's OWN label is of course always kept as its anchor.
_syn_owner = defaultdict(set)
for _p in schema_pool:
    for _s in _synonyms(_p.get("enriched_comment") or ""):
        _syn_owner[_s.lower()].add(_p["label"])
AMBIGUOUS_SYNONYMS = {s for s, owners in _syn_owner.items() if len(owners) > 1}
SCHEMA_LABEL_BY_LOWER = {p["label"].lower(): p["label"] for p in schema_pool}

def _usable_synonyms(syns, own_label):
    """Synonyms that can serve as anchors for the property labelled `own_label`."""
    if not DROP_AMBIGUOUS_SYNONYMS:
        return syns
    out = []
    for s in syns:
        k = s.lower()
        if k in AMBIGUOUS_SYNONYMS:                       # several properties claim it
            continue
        if SCHEMA_LABEL_BY_LOWER.get(k, own_label) != own_label:   # it is another property's label
            continue
        out.append(s)
    return out

def _anchors_for(item, kind, mode):
    """-> [(text, weight)] : the vectors this ONE property contributes to the pool."""
    if kind == "schema.org":
        label   = item["label"]
        blended = item.get("embedding_text") or f'{label}: {item.get("comment", "")}'
        rich    = item.get("enriched_comment") or item.get("comment") or ""
        syns    = _synonyms(rich)
    else:                                              # a grown-vocabulary property
        label   = item["canonical_label"]
        blended = vocab_embedding_text(item)
        rich    = item.get("definition") or ""
        syns    = list(item.get("aliases") or [])

    if mode == "original":
        out = [(blended, 0.0)]
    elif mode == "C":                                  # surface only — the blended vector is NOT kept
        out = [(_surface(label), 0.0)]
        out += [(_surface(s), -ANCHOR_PENALTY) for s in _usable_synonyms(syns, label)]
    else:
        out = [(blended, 0.0)]
        syns = _usable_synonyms(syns, label)
        if mode == "A":
            out.append((label, 0.0))
            out += [(s, -ANCHOR_PENALTY) for s in syns]
        elif mode == "B":
            tag = _sense_tag(_gloss(rich), label)
            out.append((f"{label}: {tag}", 0.0))
            out += [(f"{s}: {tag}", -ANCHOR_PENALTY) for s in syns]
        else:
            raise ValueError(f"unknown ANCHOR_MODE {mode!r} — use 'original', 'A', 'B' or 'C'")

    best = {}                                          # dedupe, keeping the smallest penalty
    for t, w in out:
        t = (t or "").strip()
        if t:
            best[t] = max(w, best.get(t, -9.0))
    return list(best.items()) or [(_surface(label) if mode == "C" else label, 0.0)]   # never zero anchors

def build_anchor_pool(items, kind, mode):
    """Embed every item's anchors into one matrix, remembering which rows belong to which item.
    Anchors are laid out CONTIGUOUSLY per item so the group-max is a single np.maximum.reduceat."""
    texts, offsets, starts = [], [], []
    for it in items:
        starts.append(len(texts))
        for t, w in _anchors_for(it, kind, mode):
            texts.append(t); offsets.append(w)
    starts = np.asarray(starts)
    return {"texts": texts, "starts": starts,
            "ends": np.append(starts[1:], len(texts)),
            "offsets": np.asarray(offsets, dtype="float32"),
            "emb": embed(texts, DP), "kind": kind, "mode": mode}

_ANCHOR_POOLS = {}

def schema_anchor_pool(mode):
    """Cached anchor pool over schema_pool. 'original' reuses the vectors cell 5 already embedded,
    so switching back to it costs nothing.

    The cache key includes ANCHOR_PENALTY and DROP_AMBIGUOUS_SYNONYMS because both are baked into
    the pool when it is built. Keying on `mode` alone would silently serve a stale pool after you
    changed a knob and re-ran only the comparison cell — the run would look fine and be wrong."""
    key = (mode, ANCHOR_PENALTY, DROP_AMBIGUOUS_SYNONYMS)
    if key not in _ANCHOR_POOLS:
        if mode == "original":
            n = len(schema_pool)
            _ANCHOR_POOLS[key] = {"texts": [p["embedding_text"] for p in schema_pool],
                                  "starts": np.arange(n), "ends": np.arange(1, n + 1),
                                  "offsets": np.zeros(n, dtype="float32"),
                                  "emb": schema_emb, "kind": "schema.org", "mode": mode}
        else:
            _ANCHOR_POOLS[key] = build_anchor_pool(schema_pool, "schema.org", mode)
        p = _ANCHOR_POOLS[key]
        print(f"  anchor pool [{mode}]: {len(p['texts'])} vectors for {len(schema_pool)} properties"
              f"  ({len(p['texts']) / len(schema_pool):.1f} per property)")
    return _ANCHOR_POOLS[key]

def anchor_scores(q_emb, pool):
    """-> (best, raw). best[q, item] = max over that item's (weighted) anchors — one score per
    property, exactly the contract the rest of the notebook already expects. `raw` is per-anchor,
    kept so the winning anchor can be named on the candidate record."""
    raw  = (q_emb @ pool["emb"].T) + pool["offsets"]
    best = np.maximum.reduceat(raw, pool["starts"], axis=1)
    return best, raw

def _winning_anchor(pool, raw_row, item_idx):
    """Which anchor text actually produced this property's score (None when it has only one)."""
    a, b = int(pool["starts"][item_idx]), int(pool["ends"][item_idx])
    if b - a <= 1:
        return None
    return pool["texts"][a + int(np.argmax(raw_row[a:b]))]

print(f"anchors ready | mode={ANCHOR_MODE} synonym_penalty={ANCHOR_PENALTY} "
      f"| {len(AMBIGUOUS_SYNONYMS)} ambiguous synonym(s) "
      f"{'dropped' if DROP_AMBIGUOUS_SYNONYMS else 'KEPT'}")


anchors ready | mode=original synonym_penalty=0.005 | 166 ambiguous synonym(s) dropped


In [7]:
# The growing property vocabulary store. One entry per canonical minted property:
#   id, canonical_label (camelCase property name), definition, parent (subPropertyOf: a schema.org OR
#   vocab property label, or null), domain[] / range[] (subject/object type hints), aliases[] (its
#   synonyms), provenance[], support, first_seen, and path (derived: super-property chain, stamped on save).
def load_vocab():
    if VOCAB_PATH.exists():
        return json.loads(VOCAB_PATH.read_text())
    return {"properties": [], "next_id": 1}

def save_vocab(vocab):
    _annotate_vocab_paths(vocab)                 # refresh each entry's derived `path` from its parent chain
    VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

def _sig(domain, range_):
    """domain → range for HUMAN display / the LLM judge prompt (the '→' glyph is fine there). Kept
    OUT of embedding_text — see vocab_embedding_text."""
    d = "/".join(domain) if domain else "ANY"
    r = "/".join(range_) if range_ else "ANY"
    return f"{d} → {r}"

def vocab_embedding_text(t):
    # "label: definition Also called <aliases>." — the SAME style as nb7's relation-phrase query and
    # nb5's schema property docs, so query and both document sources share one embedding style. The
    # domain -> range signature is intentionally not embedded (see nb5); it goes to the LLM judge only.
    also = f" Also called {', '.join(t['aliases'])}." if t.get("aliases") else ""
    return f'{t["canonical_label"]}: {t.get("definition", "")}{also}'

def _find_vocab_by_label(vocab, label):
    for t in vocab["properties"]:
        if t["canonical_label"] == label or label in t.get("aliases", []):
            return t
    return None

def _is_vocab_label(vocab, label):
    return _find_vocab_by_label(vocab, label) is not None

def _bump(ent, aid, phrase, instances, flip=False):
    # provenance instances are oriented to the PROPERTY's direction (swapped when flip=True), so they
    # read the same way as the stored triples.
    ent["support"] = ent.get("support", 0) + len(instances)
    def _pair(i):
        return f'{i["object_text"]} -> {i["subject_text"]}' if flip else f'{i["subject_text"]} -> {i["object_text"]}'
    ent.setdefault("provenance", []).append(
        {"article": aid, "relation_phrase": phrase, "direction": ("inverse" if flip else "as_is"),
         "instances": [_pair(i) for i in instances]})

In [8]:
# Hook A — retrieve top-k candidate properties for each relation phrase. The pool is schema.org, plus
# the grown vocabulary ONLY when INLINE_VOCAB_CANDIDATES is on (config cell). With it off the LLM never
# sees an earlier mint here, so it cannot reuse one — re-minted synonyms are collapsed later by the
# global re-consolidation instead. RERANK (config cell) adds a cross-encoder pass over the shortlist.
# Both sources expose their domain->range `signature` and their `subPropertyOf` parent, so
# the selection LLM can judge direction/domain-range fit and reach for either as a mint parent.
_SCHEMA_BY_LABEL = {p["label"]: p for p in schema_pool}

def _schema_super(label):
    """Immediate super-property (subPropertyOf) of a schema.org property, or None. schema.org lists
    sub_property_of as a (usually empty) list; we take the first schema.org super-property."""
    p = _SCHEMA_BY_LABEL.get(label)
    for sup in (p.get("sub_property_of") if p else []) or []:
        if sup in SCHEMA_LABELS:
            return sup
    return None

def _schema_super_chain(label):
    """Top-down super-property chain INCLUDING `label`: [root_super, ..., label]. Sparse for
    properties (most schema.org properties have no super-property), so usually just [label]."""
    chain, cur, seen = [], label, set()
    while cur and cur not in seen:
        seen.add(cur); chain.append(cur); cur = _schema_super(cur)
    return list(reversed(chain))

def vocab_ancestor_path(t, vocab):
    """super-property chain for a vocab property (top-down, ending at the immediate parent, EXCLUDING
    the property itself). Climbs `parent` (subPropertyOf) links up through the vocabulary, then splices
    in the schema.org super-chain of the first schema property it reaches. A `seen` guard prevents loops."""
    up, cur, seen = [], t.get("parent"), set()
    while cur and cur not in seen:
        seen.add(cur)
        ent = _find_vocab_by_label(vocab, cur)
        if ent:                                   # another vocab property -> keep climbing the vocab tree
            up.append(cur); cur = ent.get("parent")
        else:                                     # a schema.org property -> splice its super-chain in
            up.extend(reversed(_schema_super_chain(cur))); cur = None
    return " > ".join(reversed(up))

def _annotate_vocab_paths(vocab):
    """Stamp each entry's derived `path` (super-property chain, ending at the immediate parent).
    Derived from the `parent` chains (the source of truth) — refreshed on every save."""
    for t in vocab["properties"]:
        t["path"] = vocab_ancestor_path(t, vocab)
    return vocab

def _full_path(tref, vocab):
    """Full super-property path for a resolved property reference (used in the review's chosen block)."""
    if tref.get("source") == "vocab" and tref.get("vocab_id"):
        ent = next((t for t in vocab["properties"] if t["id"] == tref["vocab_id"]), None)
        if ent is not None:
            return vocab_ancestor_path(ent, vocab)
    if tref.get("source") == "schema.org" and tref.get("label"):
        return " > ".join(_schema_super_chain(tref["label"])[:-1])   # super-properties only
    return None

def retrieve_candidates(phrase_texts, vocab, mode=None, top_k=None, phrases=None, rerank=None):
    """Top-k properties per relation phrase. Each property is scored by the MAX over its anchors
    (see the anchor-pool cell), so the records handed downstream are the same shape as before —
    one score per property — plus `matched_anchor`, the anchor text that produced that score.
    `mode` defaults to ANCHOR_MODE; pass it explicitly to compare variants without re-running.
    Mode C also replaces the QUERY with the bare relation phrase (no definition): pass `phrases`
    to give it the raw phrases, else they are recovered from `phrase_texts` ("phrase: definition").

    With `rerank` (default: the RERANK flag) it fetches RERANK_FROM deep and cross-encodes down to
    top_k, so turning RERANK on in the config changes the REAL build, not just the comparison."""
    mode   = ANCHOR_MODE if mode is None else mode
    top_k  = TOP_K_MATCH if top_k is None else top_k
    rerank = RERANK if rerank is None else rerank
    fetch  = max(top_k, RERANK_FROM) if rerank else top_k
    # An article whose extraction produced NO relations reaches here with an empty phrase list.
    # embed([]) returns a 0-length 1-D array, and the anchor matmul below then dies with a core
    # dimension mismatch that names the SCHEMA POOL as the bad operand, so the traceback points at
    # the wrong thing entirely. That killed the deepseek main_test 6e run on art28 (2026-08-26),
    # whose 6c output was 788 duplicate Event hubs and zero relations. Return the empty result the
    # callers already handle. A relation-less article is still a defect worth excluding upstream —
    # this guard only stops one bad article from taking the whole stage down with it.
    if not phrase_texts:
        return []
    q_texts = phrase_texts
    if mode == "C":
        src = phrases if phrases is not None else [t.split(": ", 1)[0] for t in phrase_texts]
        q_texts = [_surface(p) for p in src]
    q_emb = embed(q_texts, QP)

    spool         = schema_anchor_pool(mode)
    sims, s_raw   = anchor_scores(q_emb, spool)
    meta          = [("schema.org", p) for p in schema_pool]
    n_schema      = len(schema_pool)
    vpool, v_raw  = None, None
    if INLINE_VOCAB_CANDIDATES and vocab["properties"]:   # grows during a build -> embedded per call
        vpool        = build_anchor_pool(vocab["properties"], "vocab", mode)
        vsims, v_raw = anchor_scores(q_emb, vpool)
        sims = np.hstack([sims, vsims])
        meta = meta + [("vocab", t) for t in vocab["properties"]]

    topk = np.argsort(-sims, axis=1)[:, :fetch]
    out = []
    for ti in range(len(phrase_texts)):
        row = []
        for j in topk[ti]:
            src, obj = meta[j]
            if src == "schema.org":
                row.append({"source": "schema.org", "label": obj["label"], "signature": obj.get("signature", ""),
                            "parent": _schema_super(obj["label"]), "inverse_of": obj.get("inverse_of"),
                            "score": round(float(sims[ti, j]), 3),
                            "matched_anchor": _winning_anchor(spool, s_raw[ti], j),
                            "comment": obj.get("enriched_comment") or obj.get("comment", "")})
            else:
                row.append({"source": "vocab", "id": obj["id"], "label": obj["canonical_label"],
                            "signature": _sig(obj.get("domain"), obj.get("range")), "parent": obj.get("parent"),
                            "inverse_of": None,
                            "score": round(float(sims[ti, j]), 3),
                            "matched_anchor": _winning_anchor(vpool, v_raw[ti], j - n_schema),
                            "comment": obj.get("definition", "")})
        out.append(row)
    if rerank:      # second stage: cross-encode the deep list down to the ten the LLM sees
        out = [rerank_candidates(q, row, top_k) for q, row in zip(phrase_texts, out)]
    return out

In [9]:
# --- Cross-encoder reranker (no training) -------------------------------------------------------
# Measured on this corpus: of the properties the pipeline eventually accepted, ~95% were reachable
# within e5's top 100 but only ~77% survived the cut at 10. The bottleneck is therefore SELECTION, not
# the embedding space — the lever is choosing a better ten, not embedding differently.
#
# A bi-encoder (e5) scores a query and a document from two vectors built independently — it never
# sees them together, which is why `recordedAt` (audio-visual capture) can outrank `hasMeasurement`
# for a temperature query. A cross-encoder reads the pair JOINTLY and scores the match directly. It
# is far too slow to run over 1,521 properties per phrase, which is exactly why it goes second:
#
#     e5 retrieves RERANK_FROM (cheap, high recall)  ->  cross-encoder cuts to the ten the LLM sees
#
# RERANK / RERANK_FROM / RERANK_MODEL come from the experiment config — do NOT redefine them here,
# or the cell would silently override the configuration the run is supposed to be reproducing.
# The default model is 22M params, English, CPU-friendly (~10-35 s for this corpus). Stronger but
# much slower on CPU: "BAAI/bge-reranker-base" (278M), "BAAI/bge-reranker-v2-m3" (568M, multilingual).

_RERANKER = None

def _reranker():
    """Loaded on first use, so the notebook costs nothing while RERANK is off."""
    global _RERANKER
    if _RERANKER is None:
        from sentence_transformers import CrossEncoder
        print(f"loading reranker {RERANK_MODEL} ...")
        _RERANKER = CrossEncoder(RERANK_MODEL)
    return _RERANKER

def rerank_candidates(query, candidates, top_k):
    """Re-order ONE phrase's candidates by cross-encoder relevance, then cut to top_k.

    The document side is "label: definition" — the definition is what lets the model separate
    schema.org's idiosyncratic senses, and it is the half mode C threw away. `score` becomes the
    cross-encoder score (a logit, so it can be negative and is NOT comparable to a cosine); the
    retrieval score and rank are kept as `embed_score` / `embed_rank` so you can see what moved."""
    if not candidates:
        return []
    pairs  = [(query, f'{c["label"]}: {(c.get("comment") or "")[:400]}') for c in candidates]
    scores = np.asarray(_reranker().predict(pairs, batch_size=64, show_progress_bar=False),
                        dtype="float64")
    out = []
    for j in np.argsort(-scores)[:top_k]:
        c = dict(candidates[int(j)])
        c["embed_score"] = c["score"]
        c["embed_rank"]  = int(j) + 1
        c["score"]       = round(float(scores[int(j)]), 4)
        out.append(c)
    return out

# --- one entry point for both the comparison and the recall diagnostic --------------------------
# A "spec" is either an anchor mode ("original", "A", "B", "C") or "rerank:<mode>", meaning
# retrieve with <mode> then cross-encode down. Keeping it a spec rather than a fifth anchor mode
# matters because reranking is a SECOND STAGE, not another way to represent the pool.
def _col(spec):
    return spec.replace(":", "_")

def candidates_for(spec, qtexts, vocab, top_k, phrases=None):
    # rerank=False everywhere: the spec decides, never the global RERANK flag, so the comparison
    # means the same thing whether or not the build is currently configured to rerank.
    if not spec.startswith("rerank:"):
        return retrieve_candidates(qtexts, vocab, mode=spec, top_k=top_k, phrases=phrases,
                                   rerank=False)
    base = spec.split(":", 1)[1]
    deep = retrieve_candidates(qtexts, vocab, mode=base, top_k=max(top_k, RERANK_FROM),
                               phrases=phrases, rerank=False)
    # rerank against the NATURAL-LANGUAGE query (phrase + definition), never mode C's bare surface
    return [rerank_candidates(q, row, top_k) for q, row in zip(qtexts, deep)]

print(f"reranker ready (lazy) | RERANK={RERANK} | {RERANK_MODEL}")
print(f"  outputs -> *{_osfx}*   |  vocab: {VOCAB_PATH.name}")


reranker ready (lazy) | RERANK=True | cross-encoder/ms-marco-MiniLM-L-6-v2
  outputs -> *_dedup_main_test*   |  vocab: nary_relation_vocabulary_dedup_main_test.json


In [10]:
# Article inputs + Gemini client (JSON mode, temperature 0 for reproducibility).
def load_article(aid):
    """Read nb6b's nary_graph.json and group relation instances by their free-form relation_phrase.
    `by_id` resolves BOTH entities (E..) and n-ary nodes (EV../OBS../CL..) to a type, so the
    subject->object signatures work for n-ary-node endpoints too."""
    g = json.loads((RESULTS_DIR / aid / f"e2e/nary_graph{_isfx}_typed_consolidated.json").read_text())
    # Sanitise the extracted graph ONCE, here, rather than guarding each of the dozens of direct
    # key reads downstream. A record the model emitted without the keys this stage indexes on
    # cannot be canonicalised at all, and letting one through kills the whole stage rather than the
    # article: qwen3-32b-think emitted a relation with no `relation_phrase` on art48 and took 6e
    # down 35 articles into an 80-article run on 2026-08-24. Dropping such a record cannot change a
    # result, because there is nothing in it to canonicalise; leaving it in costs the run.
    _drop = {"relations": [r for r in g.get("relations", []) if not r.get("relation_phrase")],
             "entities":  [e for e in g.get("entities", [])  if not e.get("id")],
             "nary_nodes": [n for n in g.get("nary_nodes", []) if not n.get("id")],
             "relation_definitions": [d for d in g.get("relation_definitions", [])
                                      if not d.get("relation_phrase")]}
    if any(_drop.values()):
        print(f"  {aid}: dropped malformed record(s) "
              + ", ".join(f"{len(v)} {k}" for k, v in _drop.items() if v))
        g["relations"]  = [r for r in g.get("relations", []) if r.get("relation_phrase")]
        g["entities"]   = [e for e in g.get("entities", [])  if e.get("id")]
        g["nary_nodes"] = [n for n in g.get("nary_nodes", []) if n.get("id")]
        g["relation_definitions"] = [d for d in g.get("relation_definitions", [])
                                     if d.get("relation_phrase")]
    by_id = {e["id"]: {"id": e["id"], "text": e.get("text"), "type": e.get("type")}
             for e in g.get("entities", [])}
    for n in g.get("nary_nodes", []):                      # n-ary nodes are first-class endpoints
        by_id[n["id"]] = {"id": n["id"], "text": n.get("label"),
                          "type": n.get("type") or n.get("node_kind")}
    rinfo   = {d["relation_phrase"]: {"definition": d.get("definition")}
               for d in g.get("relation_definitions", [])}
    phrases = list(dict.fromkeys(r["relation_phrase"] for r in g["relations"]))
    def qtext(ph):
        info = rinfo.get(ph, {})
        q = f'{ph}: {info.get("definition", "")}'.strip()
        if ALIGN_QUERY_STYLE and info.get("synonyms"):     # nary graph has no synonyms -> no-op, kept for parity
            q = f'{q} Also called {", ".join(info["synonyms"])}.'
        return q
    groups = {}
    for r in g["relations"]:
        groups.setdefault(r["relation_phrase"], []).append(r)
    return g, by_id, rinfo, phrases, [qtext(p) for p in phrases], groups

def _etype(by_id, eid):
    return (by_id.get(eid) or {}).get("type")

def group_signatures(instances, by_id):
    """Distinct observed subject_type -> object_type signatures within a relation-phrase group."""
    sigs = Counter(f'{_etype(by_id, i["subject_id"])} → {_etype(by_id, i["object_id"])}' for i in instances)
    return [s for s, _ in sigs.most_common()]

from google import genai
from google.genai import types as genai_types
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in the env and restart the kernel."
client = genai.Client(api_key=GEMINI_API_KEY)

_RETRY_STATUS = (429, 500, 502, 503, 504)
_RETRY_WORDS = ("UNAVAILABLE", "RESOURCE_EXHAUSTED", "INTERNAL", "high demand", "overloaded")

# Transport-level failures: the request never completed a round trip, so there is no server-side
# effect that repeating would duplicate. These are matched on the EXCEPTION TYPE first, because
# httpx raises ConnectError/ReadTimeout/RemoteProtocolError with messages carrying no keyword the
# list below could match, and the SDK often wraps them in something whose str() says even less.
# Without this a two-second wifi drop ends the stage — and with FRESH=True that costs the whole
# stage, not one call.
_TRANSPORT_EXC = ("ConnectError", "ConnectTimeout", "ReadTimeout", "WriteTimeout", "PoolTimeout",
                  "ReadError", "WriteError", "RemoteProtocolError", "ProtocolError",
                  "ConnectionError", "ConnectionResetError", "TimeoutError", "SSLError",
                  "IncompleteRead", "ChunkedEncodingError")
_TRANSPORT_WORDS = ("connection reset", "connection aborted", "connection refused",
                    "connection error", "server disconnected", "peer closed", "broken pipe",
                    "timed out", "timeout", "ssl", "nodename nor servname",
                    "name or service not known", "temporary failure", "network is unreachable")


# A 429 is retryable when it means "too fast" and pointless when it means "out of money": backing
# off five times against a depleted balance just burns three minutes before failing anyway.
_FATAL_WORDS = ("credits are depleted", "billing", "quota exceeded for quota metric",
                "exceeded your current quota", "insufficient")


def _gemini_retryable(exc) -> bool:
    """A transient failure — server-side OR on the wire — as opposed to one repeating cannot fix.

    Order matters: the fatal check runs first, so "out of money" is never mistaken for a transport
    hiccup and backed off against. A retried read-timeout can bill a completion twice (the server
    finished, the response was lost), which is one call — far cheaper than losing a FRESH stage.
    """
    msg = str(exc).lower()
    if any(w in msg for w in _FATAL_WORDS):
        return False
    code = getattr(exc, "code", None) or getattr(exc, "status_code", None)
    if code in _RETRY_STATUS:
        return True
    if any(w.lower() in msg for w in _RETRY_WORDS):
        return True
    # Walk the exception and its __cause__/__context__ chain, since the transport error is usually
    # wrapped rather than raised directly.
    seen, cur = set(), exc
    while cur is not None and id(cur) not in seen:
        seen.add(id(cur))
        if type(cur).__name__ in _TRANSPORT_EXC:
            return True
        cur = getattr(cur, "__cause__", None) or getattr(cur, "__context__", None)
    return any(w in msg for w in _TRANSPORT_WORDS)


def _gemini_call_with_retry(fn, attempts=6, base=4):
    """Call `fn()`, retrying TRANSIENT API errors with backoff (4s, 8s, 16s, 32s).

    Without this one `503 UNAVAILABLE - This model is currently experiencing high demand` ends the
    whole stage: it killed nb4 two articles into twenty and took nb6b and nb7b down with it. Google's
    own message says such spikes are temporary, which is what a backoff is for. A 400 or 403 still
    fails at once — repeating a malformed or unauthorised request achieves nothing.
    """
    import time
    for attempt in range(attempts):
        try:
            return fn()
        except Exception as exc:                       # noqa: BLE001 - re-raised below
            if attempt == attempts - 1 or not _gemini_retryable(exc):
                raise
            wait = min(60, base * 2 ** attempt)
            code = getattr(exc, "code", None) or getattr(exc, "status_code", "?")
            print(f"    [retry {attempt + 1}/{attempts - 1}] {type(exc).__name__} {code} — "
                  f"waiting {wait}s", flush=True)
            time.sleep(wait)


def gemini_json(system, user):
    """One JSON-mode call with DYNAMIC THINKING (thinking_budget=-1): the model decides how much to
    reason before answering. Kept on deliberately — it measurably improves the reuse-vs-mint
    decisions — at two costs worth knowing:
      * ~120k thinking tokens per build on this corpus, and thinking competes with the JSON answer
        for the output budget, so a very large article can truncate mid-string (parse_json reports
        that explicitly rather than failing cryptically). Set thinking_budget=0 if that happens.
      * run-to-run variance. temperature=0 is greedy sampling, NOT determinism: hosted inference is
        batched, so float reduction order shifts logits slightly and near-ties flip — and a sampled
        reasoning trace amplifies that. Measured on this corpus, two identical runs agreed on only
        45% of property choices (72% on schema.org-vs-mint). So a single run is not evidence: repeat
        an arm ~3x under separate RUN_TAGs and report the mean and range."""
    cfg = dict(system_instruction=system, temperature=0, response_mime_type="application/json")
    if hasattr(genai_types, "ThinkingConfig"):
        cfg["thinking_config"] = genai_types.ThinkingConfig(thinking_budget=-1)
    import time
    t0 = time.perf_counter()
    resp = _gemini_call_with_retry(lambda: client.models.generate_content(
        model=GEMINI_MODEL, contents=user, config=genai_types.GenerateContentConfig(**cfg)))
    global LAST_USAGE          # this call's tokens — now INCLUDING thinking tokens — and its latency
    LAST_USAGE = nb_parallel.record_gemini_usage(resp, time.perf_counter() - t0)
    return resp

def parse_json(resp):
    """Parse Gemini JSON from a response (or raw text), tolerating fences / trailing commas, with a
    CLEAR message on truncation instead of a cryptic JSONDecodeError."""
    # llm_backend.llm_json (the open-weight backend) already returns a PARSED object, while the
    # Gemini client above returns text. Without this, a dict falls through to getattr(resp, "text")
    # -> "" -> "empty response", which reads like the model returned nothing when it in fact
    # returned everything.
    if isinstance(resp, (dict, list)):
        return resp
    text = resp if isinstance(resp, str) else (getattr(resp, "text", "") or "")
    s = text.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\s*", "", s); s = re.sub(r"\s*```$", "", s).strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        pass
    s2 = re.sub(r",(\s*[}\]])", r"\1", s)                 # drop trailing commas before } or ]
    try:
        return json.loads(s2)
    except json.JSONDecodeError:
        pass
    try:                                                  # last resort: repair common LLM-JSON slips
        from json_repair import repair_json               # (unescaped quotes, missing commas, ...)
        return json.loads(repair_json(s))
    except Exception as e:
        fr = None
        try: fr = resp.candidates[0].finish_reason
        except Exception: pass
        try:                                              # dump the raw text so you can inspect the bad span
            _dbg = VOCAB_PATH.parent / "last_bad_llm_json.txt"; _dbg.write_text(s); _hint = f" raw saved -> {_dbg}."
        except Exception:
            _hint = ""
        truncated = "MAX_TOKENS" in str(fr)
        why = ("Output was TRUNCATED (MAX_TOKENS) — split the article." if truncated
               else "Output finished (STOP) but the JSON is malformed. Run the %pip cell to install json-repair "
                    "so the repair fallback can fix it, else inspect the saved raw JSON.")
        raise ValueError(f"Gemini JSON unparseable (finish_reason={fr}; {len(s)} chars).{_hint} {why}"
                         f"\njson error: {e}\n...tail:\n{s[-300:]}") from None


# --- select the backend ---------------------------------------------------------------------
# Must stay LAST in this cell: the Gemini `gemini_json` above is redefined every time the cell
# runs, so whichever binding happens last wins. In a separate cell this would be silently undone.
# activate() sets gemini_json, GEMINI_MODEL, RESULTS_DIR, MAX_OUTPUT_TOKENS and THINKING_BUDGET.
import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend
importlib.reload(nb_parallel); importlib.reload(llm_backend)   # plain `import` is a no-op once cached
nb_parallel.install()           # thread-aware stdout + the token/latency ledger used by usage_report

llm_backend.activate(BACKEND, ROOT, globals(), results=RESULTS_MODEL)


backend: HOSTED https://api.fireworks.ai/inference/v1 model=accounts/fireworks/models/deepseek-v4-flash-0731 thinking=False structured=True max_tokens=None temperature=caller top_p=None seed=None timeout=3600s params={'reasoning_effort': 'none'}  ->  results/deepseek-v4-flash/


{'backend': 'deepseek-v4-flash',
 'kind': 'hosted',
 'model': 'accounts/fireworks/models/deepseek-v4-flash-0731',
 'thinking': False,
 'structured': True,
 'results_dir': PosixPath('/Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/deepseek-v4-flash')}

## Inspect retrieval — top-k candidates by embedding similarity (no LLM)

Preview **Hook A on its own**: for one article, show the top-`TOP_K_MATCH` candidate properties each
relation phrase retrieves from **schema.org ∪ current property-vocabulary**, ranked by cosine
similarity — *before* the LLM judges or mints anything. On a fresh run the vocabulary is empty, so the
candidates are schema.org-only; re-run this cell after **Build** (below) to see minted `ex:` properties
surface as candidates too. No Gemini call — pure retrieval.

In [11]:
# Retrieval-only preview: the ranked candidates per relation phrase, with NO Gemini call. Uses the
# CURRENT vocab state (load_vocab(); empty => schema.org-only). This is exactly the `cands` that would
# be handed to the LLM judge in process_article — inspect it first to sanity-check what the embedder
# surfaces. Set INSPECT_ARTICLE to any id in ARTICLE_ORDER.
INSPECT_ARTICLE = ARTICLE_ORDER[0]

def preview_retrieval(aid, vocab=None, top_k=TOP_K_MATCH, show_comment=True):
    vocab = load_vocab() if vocab is None else vocab
    oie, by_id, rinfo, phrases, qtexts, groups = load_article(aid)
    cands = retrieve_candidates(qtexts, vocab)
    print(f"{aid}: {len(phrases)} relation phrases | vocab has {len(vocab['properties'])} minted "
          f"property(ies)\n" + "=" * 92)
    for i, ph in enumerate(phrases):
        sigs = ", ".join(group_signatures(groups[ph], by_id))
        print(f'\n"{ph}"   ({len(groups[ph])} instance(s) | observed {sigs})')
        print(f'   query: {qtexts[i]}')
        for k, c in enumerate(cands[i][:top_k], 1):
            inv = f'   inverse: {c["inverse_of"]}' if c.get("inverse_of") else ""
            via = f'\n          via anchor: {c["matched_anchor"]!r}' if c.get("matched_anchor") else ""
            print(f'   {k}. [{c["source"]:9}] {c["label"]:<26} score={c["score"]:.3f}   sig: {c.get("signature","")}{inv}{via}')
            if show_comment and c.get("comment"):
                print(f'          {c["comment"][:100]}')
    return cands

_ = preview_retrieval(INSPECT_ARTICLE)

  anchor pool [original]: 1521 vectors for 1521 properties  (1.0 per property)
loading reranker cross-encoder/ms-marco-MiniLM-L-6-v2 ...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

art6: 32 relation phrases | vocab has 59 minted property(ies)

"caused by"   (1 instance(s) | observed CoachDeparture → WorldCupExit)
   query: caused by: Indicates that the subject is a result of the object.
   1. [schema.org] causeOf                    score=-1.077   sig: MedicalCause → MedicalEntity   inverse: cause
          A causal relation between a medical cause and the medical entity (condition, complication, symptom) 
   2. [schema.org] source                     score=-2.404   sig: InstantaneousEvent → Thing
          A causal relation between an instantaneous event and the thing that initiated or caused it. Also cal
   3. [schema.org] cause                      score=-2.513   sig: MedicalCondition → MedicalCause   inverse: causeOf
          A causal relation between a medical condition and its medical cause. Also called "etiology", "origin
   4. [schema.org] result                     score=-3.074   sig: Action → Thing
          The outcome or product generated by an action

## Retrieval variants, side by side (no LLM calls) — *diagnostic, not part of the build*

Compares how the schema.org pool is **represented**: `original` (one blended vector per property),
`A` (+ bare surface anchors), `B` (+ surface + a short sense tag), `C` (surface forms only, on both
sides), and any `rerank:<mode>` spec (retrieve `RERANK_FROM`, cross-encode down to 10).

Writes one JSON per spec with the full top-10 per phrase, plus a single CSV with every spec across the
columns. `*_via` names the anchor that won, `*_new` flags a candidate `original` never surfaced, and
`rerank_*_was` gives the pre-rerank rank.

Independent of the build: each spec sets its own reranking, so this cell reads the same whatever
`RERANK` is set to. Skip it unless you are comparing retrieval strategies.

In [12]:
# --- Compare the retrieval specs (original / A / B / C / rerank:*), NO LLM calls ----------------
# DIAGNOSTIC, not part of the build: nothing here feeds the vocabulary, and it is independent of the
# RERANK flag (each spec sets its own), so the cell reads the same whatever the build is configured
# to do. Off by default because it re-embeds the whole property pool once per spec; flip the flag
# when you are actually comparing retrieval strategies.
#
# Same phrases, same top-k and the SAME pool for every spec: retrieval depends on the vocabulary,
# which grows during a build, so comparing specs against different pool states would measure the
# wrong thing. Writes one JSON per spec with the full top-10 records, plus one side-by-side CSV:
# one row per (article, phrase, rank), every spec across the columns.
RUN_RETRIEVAL_COMPARISON = False

if RUN_RETRIEVAL_COMPARISON:
    import csv

    COMPARE_MODES  = ["original", "A", "B", "C"]
    COMPARE_RERANK = ["original"]       # base modes to ALSO run through the cross-encoder ("rerank:X")
    COMPARE_TOP_K  = 10
    ALL_SPECS = COMPARE_MODES + [f"rerank:{m}" for m in COMPARE_RERANK]
    CMP_DIR = RESULTS_DIR / "_global" / "e2e"
    CMP_DIR.mkdir(parents=True, exist_ok=True)

    # Which document pool to compare over. "schema" isolates the question this experiment actually
    # asks — do the anchors surface the right SCHEMA.ORG property — with no grown vocabulary competing
    # for the ten slots (6j showed the vocab crowds schema.org out of a merged top-10). "current" adds
    # the saved vocabulary, reproducing the pool the selection LLM really sees mid-build.
    # Either way all three modes share ONE pool, or the comparison measures the wrong thing.
    COMPARE_VOCAB = "schema"           # "schema" -> schema.org only | "current" -> + saved vocabulary

    _cmp_vocab = {"properties": []} if COMPARE_VOCAB == "schema" else load_vocab()
    _pool_sfx  = "_schemaonly" if COMPARE_VOCAB == "schema" else ""   # never clobber the other run
    print(f"comparing {COMPARE_MODES} over {len(ARTICLE_ORDER)} article(s) | pool: "
          f"schema.org ({len(schema_pool)} properties)"
          + (f" + {len(_cmp_vocab['properties'])} minted vocab" if _cmp_vocab["properties"] else "")
          + "\n")

    per_mode = {m: {} for m in ALL_SPECS}
    rows = []
    for aid in ARTICLE_ORDER:
        oie, by_id, rinfo, phrases, qtexts, groups = load_article(aid)
        cands = {m: candidates_for(m, qtexts, _cmp_vocab, COMPARE_TOP_K, phrases=phrases)
                 for m in ALL_SPECS}
        for m in ALL_SPECS:
            per_mode[m][aid] = [{"relation_phrase": ph, "query": qtexts[i],
                                 "observed_signatures": group_signatures(groups[ph], by_id),
                                 "n_instances": len(groups[ph]), "candidates": cands[m][i]}
                                for i, ph in enumerate(phrases)]
        for i, ph in enumerate(phrases):
            seen_in_original = {c["label"] for c in cands["original"][i]}
            for r in range(COMPARE_TOP_K):
                row = {"article": aid, "relation_phrase": ph, "query": qtexts[i], "rank": r + 1}
                for m in ALL_SPECS:
                    c = cands[m][i][r] if r < len(cands[m][i]) else None
                    k = _col(m)
                    row[f"{k}_label"]  = c["label"] if c else ""
                    row[f"{k}_source"] = c["source"] if c else ""
                    row[f"{k}_score"]  = c["score"] if c else ""
                    # which anchor won, and whether this candidate is one the current pool never showed
                    row[f"{k}_via"]    = (c.get("matched_anchor") or "") if c else ""
                    row[f"{k}_new"]    = ("" if (m == "original" or not c)
                                          else ("NEW" if c["label"] not in seen_in_original else ""))
                    if c and "embed_rank" in c:      # reranked: where e5 had put it before
                        row[f"{k}_was"] = c["embed_rank"]
                rows.append(row)
        print(f"  {aid:6} {len(phrases):3} phrases")

    # --- per-mode JSON (full records: score, signature, parent, comment, winning anchor) -----------
    for m in ALL_SPECS:
        p = CMP_DIR / f"retrieval_candidates{_sfx}{_pool_sfx}_{_col(m)}.json"
        p.write_text(json.dumps({"mode": m, "top_k": COMPARE_TOP_K, "match_model": MATCH_MODEL,
                                 "rerank_model": RERANK_MODEL if m.startswith("rerank:") else None,
                                 "rerank_from": RERANK_FROM if m.startswith("rerank:") else None,
                                 "anchor_penalty": ANCHOR_PENALTY,
                                 "drop_ambiguous_synonyms": DROP_AMBIGUOUS_SYNONYMS,
                                 "compare_vocab": COMPARE_VOCAB,
                                 "vocab_size": len(_cmp_vocab["properties"]),
                                 "articles": per_mode[m]}, indent=2, ensure_ascii=False))
        print("wrote", p.relative_to(ROOT))

    # --- one side-by-side CSV ----------------------------------------------------------------------
    cols = ["article", "relation_phrase", "query", "rank"] + [
        f"{_col(m)}_{f}" for m in ALL_SPECS
        for f in (("label", "source", "score", "via", "new", "was") if m.startswith("rerank:")
                  else ("label", "source", "score", "via", "new"))]
    csv_path = CMP_DIR / f"retrieval_candidates{_sfx}{_pool_sfx}_comparison.csv"
    with csv_path.open("w", newline="", encoding="utf-8-sig") as f:      # utf-8-sig so Excel reads it
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
    print("wrote", csv_path.relative_to(ROOT), f"({len(rows)} rows)")

    # --- quick read: did anything actually move? ---------------------------------------------------
    top1 = {m: [] for m in ALL_SPECS}
    schema_share, newly = {m: 0 for m in ALL_SPECS}, {m: 0 for m in ALL_SPECS}
    n_phrases = 0
    for aid in ARTICLE_ORDER:
        for i in range(len(per_mode["original"][aid])):
            n_phrases += 1
            for m in ALL_SPECS:
                cs = per_mode[m][aid][i]["candidates"]
                top1[m].append(cs[0]["label"] if cs else None)
                schema_share[m] += sum(1 for c in cs if c["source"] == "schema.org")
                if m != "original":
                    orig = {c["label"] for c in per_mode["original"][aid][i]["candidates"]}
                    newly[m] += sum(1 for c in cs if c["label"] not in orig)

    hdr = f'{"mode":18}{"top-1 changed":>15}{"schema.org in top-10":>22}{"new candidates surfaced":>25}'
    print("\n" + hdr); print("-" * len(hdr))
    for m in ALL_SPECS:
        chg = "-" if m == "original" else sum(1 for a, b in zip(top1["original"], top1[m]) if a != b)
        print(f'{m:18}{str(chg):>15}{f"{schema_share[m]}/{n_phrases * COMPARE_TOP_K}":>22}'
              f'{("-" if m == "original" else newly[m]):>25}')
    print(f"\n{n_phrases} phrases compared. A high 'top-1 changed' means the anchors reordered the pool; "
          f"whether that is BETTER is decided by 6f-6k's mint-rate table, not by this cell.")

In [13]:
# ==== EDIT ME — the property-selection prompts, one per arm =================================
# PROPERTY_SELECT_PROMPT_MIXED / _SCHEMA are the property-side twin of nb4's TYPE selection task: canonicalize each
# free-form relation phrase to a candidate property, else mint a new ex: property.
#
# CHANGE (vs the run tagged "rr"): the first MINTING bullet used to read "name the relation the
# instances actually exhibit (their subject-type -> object-type signatures)" — which pointed at the
# signatures as naming MATERIAL, directly against the next bullet's "does NOT embed a subject or
# object … in the label". The evidence that the first bullet was winning: of the 30 object-baked
# mints in the "rr" run, 20 end in a type that literally appears as the OBJECT in that group's
# OBSERVED_SIGNATURES (Event -> Date became ex:recordDate, -> City became ex:teamHomeCity). The
# concise prompt, which lacked that parenthetical, baked far less (15-29% vs 38%).
#   The whole bullet was the problem, not just the parenthetical: "NAME the relation the instances
# exhibit" aims the naming verb at the instances, which carry entity texts AND types, so
# Heatwave -> ExcessDeathsEvent invites "causesExcessDeaths". It is now split into two steps —
# IDENTIFY (the only step that looks at the instances) and NAME (the relation alone) — with an
# explicit statement that nothing from the instances is naming material, plus a generality test:
# the label must still make sense for arguments of completely different types.
# (Relation-instance dedup is deterministic — collapsing identical subject/object pairs onto one
# triple — so it is done in code, not by the LLM; this prompt only chooses the property.)
# TWO variants, picked at call time by INLINE_VOCAB_CANDIDATES (config cell) so the prompt always
# describes the candidates the model actually receives:
#   MIXED  (INLINE_VOCAB_CANDIDATES = True)  — candidates are schema.org properties AND earlier
#          mints, marked [schema.org] / [vocab]; the model may reuse a mint, so `source` tells them apart.
#   SCHEMA (INLINE_VOCAB_CANDIDATES = False) — candidates are schema.org ONLY. The vocabulary is
#          deliberately hidden, so the prompt tells the model not to guess at unseen mints and that
#          re-minting across documents is expected (the global re-consolidation collapses duplicates).
PROPERTY_SELECT_PROMPT_MIXED = """You are canonicalizing free-form relation phrases against a shared property vocabulary that is reused across many documents.
You are given several GROUPS from one news document. Each group is one free-form relation phrase, read as subject --phrase--> object, with the instances that were given it. Assign every instance a property, and never compare or merge instances across groups.

Each group provides:
- RELATION_PHRASE: the free-form relation label, plus its DEFINITION if any.
- OBSERVED_SIGNATURES: the distinct subject-type → object-type pairs seen among the instances.
- CANDIDATES: a ranked list of existing properties proposed as matches — each a standard schema.org property marked [schema.org], or one minted for an earlier document marked [vocab] — with its comment, its `signature` written domain → range (a "/" lists alternative types, ANY means unconstrained), its `subPropertyOf`, and its `inverse` where schema.org names one.
- INSTANCES: the numbered subject --> object pairs, with their entity types.

An assignment is a subset of a group's instances plus the one property chosen for them. Put the whole group in one assignment by default, and split only when instances genuinely fit different properties — usually because their signatures differ. Every instance belongs to exactly one assignment.

For each assignment, choose the property in this order:
1. as_is: a candidate that carries the same relation and already runs subject --> object. On a tie, take the [schema.org] one.
2. inverse: no candidate runs subject --> object, but one expresses object --> subject: reuse it with direction "inverse" rather than mint a redundant property, and the triple is stored swapped to match the property's own direction. If a candidate names an `inverse` that is itself listed and fits better, select that one as_is.
3. new: mint, only when neither reading of any candidate fits.
Judge direction from the signatures and the instance types, not from comment wording: comments are written neutrally and may not state a direction.

Accepting a candidate: judge on the INSTANCES, not the phrase definition alone — does the candidate express the same relation for these pairs? Match their subject -> object against its domain -> range for as_is, or against its range -> domain for inverse. Domain and range are permissive suggestions and our entity types are routinely subtypes, supertypes or siblings of them, so accept whenever the meaning is preserved and the arguments are broadly compatible; reject on type grounds only when the pairing is meaningless, or would assert a different relation.

Minting: mint only when reusing the best candidate would change or lose the relation's meaning, including when it fits only by being so broad that the distinguishing meaning is gone. 
- Identify the relation, then name it. Only the first step looks at the instances: they tell you which relation you are naming, never what to call it.
- The name is a concise, reusable camelCase label for the relation alone, as if you were adding it to a standard vocabulary. Neither a subject nor an object may be baked into it, neither its type nor a specific entity: "after", not "afterEvent"; "locatedIn", not "locatedInCity"; "hasValue", not "hasQuantityValue". Two phrasings of one relation must give one label, and the label must still read correctly for arguments of completely different types.
- Give a one-sentence definition, a parent (the closest existing property it should be a subPropertyOf — a candidate label or any candidate's own subPropertyOf — or null if none is sensible), and domain / range as short type lists drawn from the OBSERVED_SIGNATURES. A mint is always in the phrase's own direction, so its direction is "as_is".

Return only valid JSON, one object per group, in the input order, covering every instance number exactly once:
{"results": [{"relation_phrase": "<phrase>", "assignments": [{"instances": [1, 2], "chosen_property": {"source": "schema.org | vocab | new", "label": "<property label>", "is_new": false, "direction": "as_is", "definition": null, "parent": null, "domain": null, "range": null, "justification": "one short line"}}]}]}
`label` is the exact candidate label or your new one, never a paraphrase. When you reuse, `source` echoes that candidate's marked source and definition / parent / domain / range stay null; when you mint, set `source` to "new" and `is_new` true, and fill them. `justification` is one line, saying so when you flipped to inverse."""

PROPERTY_SELECT_PROMPT_SCHEMA = """You are canonicalizing free-form relation phrases against schema.org, minting a new property only where no schema.org property fits.
You are given several GROUPS from one news document. Each group is one free-form relation phrase, read as subject --phrase--> object, with the instances that were given it. Assign every instance a property, and never compare or merge instances across groups.

Each group provides:
- RELATION_PHRASE: the free-form relation label, plus its DEFINITION if any.
- OBSERVED_SIGNATURES: the distinct subject-type → object-type pairs seen among the instances.
- CANDIDATES: a ranked list of schema.org properties proposed as matches, each with its comment, its `signature` written domain → range (a "/" lists alternative types, ANY means unconstrained), its `subPropertyOf`, and its `inverse` where schema.org names one.
- INSTANCES: the numbered subject --> object pairs, with their entity types.

An assignment is a subset of a group's instances plus the one property chosen for them. Put the whole group in one assignment by default, and split only when instances genuinely fit different properties — usually because their signatures differ. Every instance belongs to exactly one assignment.

For each assignment, choose the property in this order:
1. as_is: a candidate that carries the same relation and already runs subject --> object.
2. inverse: no candidate runs subject --> object, but one expresses object --> subject: reuse it with direction "inverse" rather than mint a redundant property, and the triple is stored swapped to match the property's own direction. If a candidate names an `inverse` that is itself listed and fits better, select that one as_is.
3. new: mint, only when neither reading of any candidate fits.
Judge direction from the signatures and the instance types, not from comment wording: comments are written neutrally and may not state a direction.

Accepting a candidate: judge on the INSTANCES, not the phrase definition alone — does the candidate express the same relation for these pairs? Match their subject -> object against its domain -> range for as_is, or against its range -> domain for inverse. Domain and range are permissive suggestions and our entity types are routinely subtypes, supertypes or siblings of them, so accept whenever the meaning is preserved and the arguments are broadly compatible; reject on type grounds only when the pairing is meaningless, or would assert a different relation.

Minting: mint only when reusing the best candidate would change or lose the relation's meaning, including when it fits only by being so broad that the distinguishing meaning is gone.
- Identify the relation, then name it. Only the first step looks at the instances: they tell you which relation you are naming, never what to call it.
- The name is a concise, reusable camelCase label for the relation alone, as if you were adding it to a standard vocabulary. Neither a subject nor an object may be baked into it, neither its type nor a specific entity: "after", not "afterEvent"; "locatedIn", not "locatedInCity"; "hasValue", not "hasQuantityValue". Two phrasings of one relation must give one label, and the label must still read correctly for arguments of completely different types.
- Give a one-sentence definition, a parent (the closest existing property it should be a subPropertyOf — a candidate label or any candidate's own subPropertyOf — or null if none is sensible), and domain / range as short type lists drawn from the OBSERVED_SIGNATURES. A mint is always in the phrase's own direction, so its direction is "as_is".

Return only valid JSON, one object per group, in the input order, covering every instance number exactly once:
{"results": [{"relation_phrase": "<phrase>", "assignments": [{"instances": [1, 2], "chosen_property": {"source": "schema.org | new", "label": "<property label>", "is_new": false, "direction": "as_is", "definition": null, "parent": null, "domain": null, "range": null, "justification": "one short line"}}]}]}
`label` is the exact candidate label or your new one, never a paraphrase. When you reuse, `source` is "schema.org" and definition / parent / domain / range stay null; when you mint, set `source` to "new" and `is_new` true, and fill them. `justification` is one line, saying so when you flipped to inverse."""

# ==== ALTERNATIVE arm: CONCISE ==============================================================
# Same task and the SAME output schema as the prompts above — so the caller, the review files and
# everything downstream are unchanged — but each rule is stated once. The long version repeats
# "prefer reuse over minting" three times (decision procedure, instance test, minting), explains the
# soft domain/range treatment twice, spreads direction guidance over four places, and describes the
# output shape in prose as well as in the JSON skeleton. This keeps every RULE and drops the
# restatements: ~3.4k chars vs ~7.2k, i.e. ~1k fewer prompt tokens per article.
#
# v2 (the text below) is a MEASURED revision. v1 was this prompt with every rule stated exactly once.
# On the same 6d input it did worse than the long prompt: 95 vs 76 minted properties after
# consolidation, 62.6% vs 70.2% schema.org reuse, and it cost MORE tokens (407k vs 383k), because
# more minting means longer candidate lists and longer mint records. Reading that as "the repetition
# was load-bearing", v2 puts back the reuse-over-mint emphasis — and ONLY that — in the three places
# the long prompt states it, leaving every other rule stated once:
#   1. "reusing an existing property is strongly preferred over minting"  (decision procedure)
#   2. "a broader candidate that preserves what is distinctive beats a new property"  (type fit)
#   3. "do NOT mint merely because the types are not an exact match"  (minting)
# v1's run is preserved on disk under RUN_TAG="rr_concise" (graphs, review files, token usage).
#
# It was authored for a MIXED candidate list, so it is used as-is under either inline pool.
PROPERTY_SELECT_PROMPT_CONCISE = """You canonicalize relation phrases to properties. You are given groups of relation instances from one document; each group shares one free-form relation phrase. Treat every group independently — never merge or compare instances across groups.

INPUT, per group:
- RELATION_PHRASE + its definition. A relation reads subject --RELATION_PHRASE--> object.
- OBSERVED_SIGNATURES: the subject-type -> object-type pairs actually seen among the instances.
- CANDIDATES: ranked properties, each marked [schema.org] or [vocab], with its `signature` (domain -> range) and its `parent` (super-property, if any).
- INSTANCES: a NUMBERED list of subject --> object pairs with their entity types.

OUTPUT, per group: one or more ASSIGNMENTS. An assignment is a set of instance NUMBERS plus the single property chosen for them. Every instance belongs to exactly one assignment. Default to ONE assignment covering the whole group; split only when instances genuinely need different properties because their subject/object types differ.

CHOOSING A PROPERTY — reusing an existing property is strongly preferred over minting. Take the first option that works:
1. "as_is"   — a candidate with the same relation meaning that already runs subject --> object.
2. "inverse" — a candidate expressing that same relation as object --> subject. Choosing it swaps subject and object so the stored triple runs in the property's own direction. If a candidate lists `inverse: <name>` and that named property is itself in the list, pick that one directly as_is instead.
3. "new"     — mint, only when neither reading of any candidate preserves the relation's meaning.
When a [schema.org] and a [vocab] candidate fit equally well, take the schema.org one.

DIRECTION is judged from the signatures and the instance types, never from candidate comments — comments are written neutrally and state no direction.

DOMAIN/RANGE IS A SOFT SIGNAL. Our entity types are routinely subtypes, supertypes or siblings of a property's declared domain/range. Accept a candidate whenever it preserves the relation's meaning and the arguments are broadly compatible. Reject on type grounds only when the pairing is meaningless, or would make the property assert a different KIND of relation. A candidate that is somewhat broader but still preserves what makes the relation distinctive is preferable to a new property.

MINT ONLY IF reuse would change or lose what makes this relation distinctive — including when the only candidate that fits is so broad it says nothing. Do NOT mint merely because the subject/object types are not an exact match for a candidate's declared domain/range: a meaning-preserving candidate with a loose type fit beats a new property. When minting:
- Name the relation the INSTANCES actually exhibit; this may differ from a vague or misfitting relation phrase.
- Use a general, reusable camelCase label that names the relation and NEVER embeds a subject or object, neither its type nor a specific entity: "after" not "afterEvent", "locatedIn" not "locatedInCity", "hasValue" not "hasQuantityValue". Two phrasings of one relation must produce the same label.
- Direction is always "as_is". Give a one-sentence definition, a parent (the closest candidate label or candidate `parent` that is a sensible super-property, else null), and domain / range taken from OBSERVED_SIGNATURES.

Return ONLY valid JSON — no prose, no code fences — one result object per input group, in the same order, each echoing its exact relation_phrase. Angle brackets show shape, not literal values; definition / parent / domain / range are null unless minting.
{"results": [{"relation_phrase": "<phrase>", "assignments": [{"instances": [1, 2], "chosen_property": {"source": "schema.org | vocab | new", "label": "<property label>", "is_new": false, "direction": "as_is", "definition": null, "parent": null, "domain": null, "range": null, "justification": "one short line"}}]}]}
Every group appears exactly once; every instance NUMBER appears exactly once across its group's assignments."""


def active_select_prompt():
    """The prompt this run actually sends: the config's `prompt` arm, and — for the long arm — the
    variant matching the candidates retrieval really produces (see INLINE_VOCAB_CANDIDATES)."""
    if PROMPT == "concise":
        return PROPERTY_SELECT_PROMPT_CONCISE
    return PROPERTY_SELECT_PROMPT_MIXED if INLINE_VOCAB_CANDIDATES else PROPERTY_SELECT_PROMPT_SCHEMA

assert PROMPT in ("long", "concise"), f"config prompt={PROMPT!r}; use 'long' or 'concise'"
print(f"property-selection prompt: {PROMPT.upper()} · "
      + ("MIXED (schema.org + vocab)" if INLINE_VOCAB_CANDIDATES
         else "SCHEMA-ONLY (vocabulary hidden from the LLM)")
      + f"  ({len(active_select_prompt()):,} chars)")


property-selection prompt: LONG · SCHEMA-ONLY (vocabulary hidden from the LLM)  (4,327 chars)


In [14]:
# Hook A caller — ONE Gemini call per article. For each relation-phrase group the LLM returns one or
# more ASSIGNMENTS: each assignment is a subset of the group's instances (by 1-based number) plus the
# property chosen for that subset. This lets a single phrase split across several properties when its
# instances have different signatures. Returns: phrase -> [{"instances": [0-based idx...], "chosen": {...}}].
def llm_assign(phrases, rinfo, groups, by_id, cands):
    blocks = []
    for i, ph in enumerate(phrases):
        b = [f"GROUP {i+1} —", f'RELATION_PHRASE: "{ph}"']
        if rinfo.get(ph, {}).get("definition"):
            b.append(f'DEFINITION: {rinfo[ph]["definition"]}')
        b.append("OBSERVED_SIGNATURES: " + "; ".join(group_signatures(groups[ph], by_id)))
        b.append("CANDIDATES (ranked):")
        if cands[i]:
            for k, c in enumerate(cands[i], 1):
                par = c.get("parent") or "—"
                inv = f' | inverse: {c["inverse_of"]}' if c.get("inverse_of") else ""
                b.append(f'  {k}. [{c["source"]}] {c["label"]} | signature: {c.get("signature","")} | subPropertyOf: {par}{inv}')
                b.append(f'     {c.get("comment", "")}')
        else:
            b.append("  (none)")
        b.append("INSTANCES (numbered):")
        b += [f'  {j}. "{r["subject_text"]}" ({_etype(by_id, r["subject_id"])}) --> "{r["object_text"]}" ({_etype(by_id, r["object_id"])})'
              for j, r in enumerate(groups[ph], 1)]
        blocks.append("\n".join(b))
    user = f"You are given {len(phrases)} GROUPS from one document.\n\n" + "\n\n".join(blocks)
    parsed  = parse_json(gemini_json(active_select_prompt(), user))
    results = parsed.get("results", parsed) if isinstance(parsed, dict) else parsed
    # An open-weight model in SOFT-JSON mode (thinking arms: no grammar constraint) sometimes
    # answers `"chosen_type": "Person"` — a bare string where the schema says an object — and
    # occasionally drops a bare string into `results` too. `or {}` kept the string, which then died
    # ~200 lines later on `ch.get(...)`, after the whole chunk's LLM calls had already been paid for.
    # Coerce at the parse boundary: a bare label is exactly what the no-match fallback path expects,
    # and a malformed row is skipped rather than being fatal.
    def _as_choice(v):
        if isinstance(v, str):
            return {"label": v}
        return v if isinstance(v, dict) else {}

    raw = {r.get("relation_phrase"): r for r in (results or [])
           if isinstance(r, dict) and r.get("relation_phrase") in set(phrases)}

    out = {}
    for i, ph in enumerate(phrases):
        n   = len(groups[ph])
        top = cands[i][0] if cands[i] else {"source": "schema.org", "label": "relatedTo"}
        fallback = {"source": top["source"], "label": top["label"], "is_new": False,
                    "direction": "as_is", "justification": "fallback: top candidate"}
        asgs, seen = [], set()
        for a in ((raw.get(ph) or {}).get("assignments") or []):
            ch, idxs = _as_choice(a.get("chosen_property")), []
            for x in (a.get("instances") or []):
                try:
                    j = int(x) - 1
                except (TypeError, ValueError):
                    continue
                if 0 <= j < n and j not in seen:          # valid, first assignment wins on any overlap
                    seen.add(j); idxs.append(j)
            if idxs:
                asgs.append({"instances": idxs, "chosen": ch})
        missing = [j for j in range(n) if j not in seen]  # any uncovered instance -> fallback assignment
        if missing:
            asgs.append({"instances": missing, "chosen": fallback})
        out[ph] = asgs or [{"instances": list(range(n)), "chosen": fallback}]
    return out

# ---- Minting: admit a chosen "new" property into the vocabulary -------------------------------
def _schema_ref(label):
    return {"label": label, "source": "schema.org", "vocab_id": None,
            "definition": None, "parent": None, "domain": None, "range": None, "is_new_to_vocab": False}

def _vocab_ref(ent, is_new=False):
    return {"label": ent["canonical_label"], "source": "vocab", "vocab_id": ent["id"],
            "definition": ent["definition"], "parent": ent["parent"],
            "domain": ent.get("domain"), "range": ent.get("range"), "is_new_to_vocab": is_new}

# Admit a minted property as a NEW vocabulary entry (or, in restrict mode, fall back to its parent /
# a generic related-to link). Parent (subPropertyOf) may be null — properties have no single root.
def _apply_mint(aid, phrase, ch, group, vocab):
    label = ch.get("label"); definition = ch.get("definition") or ""
    parent = ch.get("parent")
    if parent is not None and not (parent in SCHEMA_LABELS or _is_vocab_label(vocab, parent)):
        parent = None                                          # unknown parent -> no super-property
    if not EXPAND_MODE:                                        # restrict mode: fall back to the parent
        return _schema_ref(parent) if parent in SCHEMA_LABELS else _schema_ref("relatedTo")
    ent = {"id": f"P{vocab['next_id']:04d}", "canonical_label": label, "definition": definition,
           "parent": parent, "domain": ch.get("domain") or [], "range": ch.get("range") or [],
           "aliases": [], "support": len(group), "first_seen": aid,
           "provenance": [{"article": aid, "relation_phrase": phrase,
                           "instances": [f'{i["subject_text"]} -> {i["object_text"]}' for i in group]}]}
    vocab["properties"].append(ent); vocab["next_id"] += 1
    return _vocab_ref(ent, is_new=True)

In [15]:
# Build canonical relation records from per-instance assignments. `inst_assignments` is a flat list of
# (phrase, instance, tref) — each instance carries the property (tref) chosen for ITS assignment, so two
# instances of the same phrase can land on different properties. Each instance is ORIENTED to its
# property's direction (subject/object SWAPPED when tref direction == "inverse"), then instances are
# DEDUPED deterministically on (subject_id, property, object_id): identical triples (e.g. a direct and an
# inverse phrase that resolve to the same property between the same two entities) collapse into ONE
# record, evidence unioned. No LLM here.
def _build_relation_records(by_id, inst_assignments):
    records = {}   # (subject_id, property_label, object_id) -> record
    for ph, r, tref in inst_assignments:
        flip = tref.get("direction") == "inverse"
        if flip:                                                  # swap to the property's direction
            s_id, s_txt, o_id, o_txt = r["object_id"], r["object_text"], r["subject_id"], r["subject_text"]
        else:
            s_id, s_txt, o_id, o_txt = r["subject_id"], r["subject_text"], r["object_id"], r["object_text"]
        key = (s_id, tref.get("label"), o_id)
        rec = records.get(key)
        if rec is None:
            rec = {
                "subject_id":   s_id,
                "subject_text": s_txt,
                "subject_type": _etype(by_id, s_id),
                "property":                 tref.get("label"),
                "property_source":          tref.get("source"),
                "vocab_id":                 tref.get("vocab_id"),
                "property_is_new_to_vocab": tref.get("is_new_to_vocab", False),
                "property_definition":      tref.get("definition"),
                "property_parent":          tref.get("parent"),
                "property_justification":   tref.get("justification"),
                "object_id":    o_id,
                "object_text":  o_txt,
                "object_type":  _etype(by_id, o_id),
                "relation_phrases": [],          # the free-form phrase(s) that produced this triple
                "phrase_directions": {},         # phrase -> "as_is" | "inverse" (was it flipped?)
                "evidence":         [],
                "confidence":       0.0,
            }
            records[key] = rec
        if ph not in rec["relation_phrases"]:
            rec["relation_phrases"].append(ph)
        rec["phrase_directions"][ph] = "inverse" if flip else "as_is"
        ev = r.get("evidence")
        if ev and ev not in rec["evidence"]:
            rec["evidence"].append(ev)
        rec["confidence"] = max(rec["confidence"], float(r.get("confidence") or 0.0))
    return list(records.values())

In [16]:
# Process ONE article end-to-end, self-contained:
#   retrieve (Hook A) -> ONE LLM call assigning each instance a property (a phrase may split across
#   several properties) -> admit any minted properties as new vocabulary entries -> build canonical
#   (deduped) relation records. Writes nary_graph_ext.json AND nary_relation_matching_review.json (per
#   relation phrase: query text, the candidates offered, and EACH assignment's instance subset + chosen property).
def prefetch_assignments(aid, vocab):
    """Everything up to AND INCLUDING the LLM call, mutating nothing. `vocab` is read only by
    retrieve_candidates, which ignores it when INLINE_VOCAB_CANDIDATES is False — which is exactly
    why these may run in parallel then. Token usage is captured per THREAD, so each article keeps its
    own counts even with several calls in flight."""
    oie, by_id, rinfo, phrases, qtexts, groups = load_article(aid)
    cands  = retrieve_candidates(qtexts, vocab)
    report = llm_assign(phrases, rinfo, groups, by_id, cands)   # phrase -> [assignments]
    return {"oie": oie, "by_id": by_id, "rinfo": rinfo, "phrases": phrases, "qtexts": qtexts,
            "groups": groups, "cands": cands, "report": report, "usage": nb_parallel.last_usage()}

def process_article(aid, vocab, write=True, save=True, verbose=True, pre=None):
    # `pre` is a prefetch_assignments() result. Everything BELOW this point mutates the vocabulary
    # (mint / bump / save) and must therefore run one article at a time, in ARTICLE_ORDER.
    if pre is None:
        pre = prefetch_assignments(aid, vocab)
    oie, by_id, rinfo = pre["oie"], pre["by_id"], pre["rinfo"]
    phrases, qtexts, groups = pre["phrases"], pre["qtexts"], pre["groups"]
    cands, report = pre["cands"], pre["report"]

    assigns = {}                                 # phrase -> [{"instances": [r...], "tref": tref, "chosen": ch}]
    inst_res, n_mint, n_flip = [], 0, 0
    for ph in phrases:
        rows = []
        for a in report[ph]:
            ch    = a["chosen"]
            insts = [groups[ph][j] for j in a["instances"]]
            is_mint  = ch.get("is_new") or ch.get("source") == "new"
            existing = _find_vocab_by_label(vocab, ch.get("label"))   # reuse an existing entry if the label matches
            # a mint is in the phrase's own direction; a reused candidate may be flipped to inverse
            direction = "as_is" if (is_mint and not existing) else ("inverse" if ch.get("direction") == "inverse" else "as_is")
            if is_mint and not existing:                              # admit a genuinely new property
                before = len(vocab["properties"])
                tref = _apply_mint(aid, ph, ch, insts, vocab)
                n_mint += (len(vocab["properties"]) > before)
            elif existing:                                            # chose (or minted a dup of) an existing vocab property
                _bump(existing, aid, ph, insts, flip=(direction == "inverse")); tref = _vocab_ref(existing)
            else:                                                     # a schema.org candidate
                tref = _schema_ref(ch.get("label"))
            tref["direction"] = direction
            tref["justification"] = ch.get("justification")
            n_flip += (direction == "inverse") * len(insts)
            for r in insts:
                inst_res.append((ph, r, tref))
            rows.append({"instances": insts, "tref": tref, "chosen": ch})
        assigns[ph] = rows

    records = _build_relation_records(by_id, inst_res)

    if write:
        out_dir = RESULTS_DIR / aid
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / f"e2e/nary_graph{_osfx}_extended.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE,
             "entities": oie.get("entities", []), "nary_nodes": oie.get("nary_nodes", []),
             "relations": records},
            indent=2, ensure_ascii=False))
        # Review file: one row per relation phrase, listing EACH assignment (its instance subset + the
        # property chosen for it, with super-property path) so a phrase split across properties is fully
        # auditable, plus the candidates the LLM chose from at THIS article's vocab state.
        def _chosen_block(tref):
            return {"label": tref.get("label"), "source": tref.get("source"), "direction": tref.get("direction"),
                    "vocab_id": tref.get("vocab_id"), "is_new_to_vocab": tref.get("is_new_to_vocab", False),
                    "definition": tref.get("definition"), "parent": tref.get("parent"),
                    "domain": tref.get("domain"), "range": tref.get("range"),
                    "path": _full_path(tref, vocab), "justification": tref.get("justification")}
        review = [{
            "relation_phrase":      ph,
            "definition":           rinfo.get(ph, {}).get("definition"),
            "embedding_query_text": qtexts[i],
            "observed_signatures":  group_signatures(groups[ph], by_id),
            "n_assignments":        len(assigns[ph]),
            "assignments": [{
                "instances":       [f'{r["subject_text"]} -> {r["object_text"]}' for r in a["instances"]],
                "chosen_property": _chosen_block(a["tref"]),
                "llm_proposed":    {k: (a["chosen"] or {}).get(k) for k in ("source", "label", "is_new", "direction")},
            } for a in assigns[ph]],
            "candidates":           cands[i],
        } for i, ph in enumerate(phrases)]
        (out_dir / f"e2e/nary_relation_matching_review{_osfx}.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE, "relations": review},
            indent=2, ensure_ascii=False))
    if save:
        save_vocab(vocab)
    if verbose:
        merged  = sum(1 for r in records if len(r["relation_phrases"]) > 1 or len(r["evidence"]) > 1)
        n_split = sum(1 for ph in phrases if len(assigns[ph]) > 1)
        n_inst  = len(oie["relations"])
        print(f"  {aid}: {len(phrases)} phrases ({n_split} split), {n_inst} instances -> {len(records)} triples "
              f"({merged} merged, {n_flip} inverse-reused) | +{n_mint} minted | vocab now {len(vocab['properties'])}")
    return records, vocab

## Dry run — one article, no writes

In [17]:
# Dry run on the FIRST article against a COPY of the current vocabulary: one real LLM call, but no
# writes and no vocabulary mutation, so it is safe to fire before committing to a full build.
DRY_RUN = False

if DRY_RUN:
    _vocab_preview = copy.deepcopy(load_vocab())
    _records, _ = process_article(ARTICLE_ORDER[0], _vocab_preview, write=False, save=False)
    for (src, lbl), c in Counter((r.get("property_source"), r["property"]) for r in _records).most_common():
        print(f"  {str(src):<11} {lbl}  x{c}")

## Build — incremental vocabulary over all articles (in order)

In [18]:
# Incremental build.
#
# FRESH comes from the config. True: start from an EMPTY vocabulary and process ALL of
# ARTICLE_ORDER — what a reported run does, reproducible from the config alone. False: keep the
# existing nary_relation_vocabulary<_osfx>.json and extend it with NEW_ARTICLES only (or, if that is
# empty, with every article that has no extended graph yet).
NEW_ARTICLES = []          # used only when FRESH is False

if FRESH:
    vocab = {"properties": [], "next_id": 1}; save_vocab(vocab); todo = ARTICLE_ORDER
else:
    vocab = load_vocab()
    todo  = NEW_ARTICLES or [a for a in ARTICLE_ORDER
                             if not (RESULTS_DIR / a / f"e2e/nary_graph{_osfx}_extended.json").exists()]

# CONSOLIDATE_EVERY (config, 0 = off): re-consolidate every N articles instead of once at the end.
# This cell then builds only the FIRST chunk — consolidate_vocab / refresh_ext_after_consolidation
# are defined further down, so the rest is driven by the periodic cell below them.
TODO_ALL = list(todo)
if CONSOLIDATE_EVERY:
    todo = TODO_ALL[:CONSOLIDATE_EVERY]

print(f"building property vocabulary over {len(todo)} article(s): {todo}\n")
nb_parallel.reset_usage()          # the report below covers only the calls THIS run makes
_t0 = time.perf_counter()

LAST_USAGE = None
TOKENS = []

# Phase 1 — the LLM calls, in parallel when they are independent of the vocabulary (see PARALLEL_LLM
# in the config cell). Nothing is applied here, so the vocabulary is untouched.
_pre = {}
if PARALLEL_LLM and INLINE_VOCAB_CANDIDATES:
    print("PARALLEL_LLM ignored: INLINE_VOCAB_CANDIDATES is True, so each article's candidate list\n"
          "depends on the vocabulary the earlier articles built — the calls are NOT independent.\n")
elif PARALLEL_LLM and len(todo) > 1:
    print(f"phase 1 — {len(todo)} LLM calls on {WORKERS} threads (inline pool is schema.org only)")
    _pre = {r.item: r.value for r in
            nb_parallel.pmap(lambda a: prefetch_assignments(a, vocab), todo, workers=WORKERS)
            if r.ok}
    print("\nphase 2 — applying to the vocabulary in ARTICLE_ORDER (no LLM calls)")

# Phase 2 — apply, strictly in ARTICLE_ORDER, so the vocabulary evolves exactly as it would in a
# sequential run and the written output is identical; only the API waiting was overlapped.
for aid in todo:
    _ta = time.perf_counter()
    with nb_parallel.usage_scope(aid):      # tag this article's calls so the report breaks down per article
        process_article(aid, vocab, write=True, save=True, verbose=True, pre=_pre.get(aid))
    _u = (_pre.get(aid) or {}).get("usage") or LAST_USAGE   # prefetched -> that article's OWN counts
    TOKENS.append({"article": aid, "seconds": round(time.perf_counter() - _ta, 1),
                   **(_u or {})})           # tokens (incl. THINKING) + elapsed for that article
    if aid not in _pre:                     # nothing to pace when the call already happened
        time.sleep(5)
print(f"\ndone — {len(vocab['properties'])} canonical properties in {VOCAB_PATH}")

# Per-article token log, tagged per experiment so each run keeps its own (run_metrics reads it).
_toktot = {k: sum((t.get(k) or 0) for t in TOKENS)
           for k in ("prompt_tokens", "output_tokens", "thoughts_tokens", "total_tokens", "seconds")}
_tokp = RESULTS_DIR / "_global" / "e2e" / f"token_usage{_osfx}.json"
_tokp.write_text(json.dumps({"method": "candidate", "experiment": EXPERIMENT, "variant": VARIANT,
                             "prompt": PROMPT, "model": GEMINI_MODEL,
                             "per_article": TOKENS, "totals": _toktot}, indent=2, ensure_ascii=False))
print(f"tokens (candidate): {_toktot} -> {_tokp.name}")

# --- what this run cost: calls, prompt / output / THINKING tokens, and elapsed time ------------
nb_parallel.usage_report(f"6e · e2e relation canonicalization · {EXPERIMENT}",
                         wall_seconds=time.perf_counter() - _t0,
                         experiment=EXPERIMENT, run_tag=RUN_TAG)

building property vocabulary over 5 article(s): ['art6', 'art7', 'art9', 'art10', 'art11']

phase 1 — 5 LLM calls on 5 threads (inline pool is schema.org only)
5 item(s), running on 5 threads

[1/5] art11  (56.7s)
[2/5] art9  (65.7s)
[3/5] art6  (74.3s)
[4/5] art7  (80.2s)
[5/5] art10  (80.8s)

done — 5/5 ok in 80.8s

phase 2 — applying to the vocabulary in ARTICLE_ORDER (no LLM calls)
  art6: 32 phrases (0 split), 52 instances -> 52 triples (0 merged, 11 inverse-reused) | +4 minted | vocab now 4
  art7: 38 phrases (0 split), 70 instances -> 70 triples (0 merged, 5 inverse-reused) | +4 minted | vocab now 8
  art9: 27 phrases (1 split), 84 instances -> 84 triples (0 merged, 17 inverse-reused) | +5 minted | vocab now 13
  art10: 36 phrases (0 split), 361 instances -> 361 triples (0 merged, 0 inverse-reused) | +20 minted | vocab now 33
  art11: 21 phrases (0 split), 59 instances -> 59 triples (0 merged, 2 inverse-reused) | +0 minted | vocab now 33

done — 33 canonical properties in /Users

{'label': '6e · e2e relation canonicalization · deepseek-loop-drop',
 'totals': {'calls': 5,
  'prompt_tokens': 187501,
  'output_tokens': 17181,
  'thoughts_tokens': 0,
  'total_tokens': 204682,
  'seconds': 108.0},
 'wall_seconds': 80.89499879200594,
 'per_tag': {'art11': {'calls': 1,
   'seconds': 16.3,
   'prompt_tokens': 25554,
   'output_tokens': 2279,
   'thoughts_tokens': 0,
   'total_tokens': 27833},
  'art9': {'calls': 1,
   'seconds': 18.2,
   'prompt_tokens': 33025,
   'output_tokens': 3059,
   'thoughts_tokens': 0,
   'total_tokens': 36084},
  'art6': {'calls': 1,
   'seconds': 21.9,
   'prompt_tokens': 37355,
   'output_tokens': 3319,
   'thoughts_tokens': 0,
   'total_tokens': 40674},
  'art7': {'calls': 1,
   'seconds': 25.2,
   'prompt_tokens': 43066,
   'output_tokens': 3802,
   'thoughts_tokens': 0,
   'total_tokens': 46868},
  'art10': {'calls': 1,
   'seconds': 26.4,
   'prompt_tokens': 48501,
   'output_tokens': 4722,
   'thoughts_tokens': 0,
   'total_tokens': 53

## Report + verify

In [19]:
# Property vocabulary summary, then verify each ext file covers every nb6 relation instance
# (each oie relation maps to exactly one canonical triple — none lost; merges only collapse duplicates).
vocab = load_vocab()
print(f"PROPERTY VOCABULARY — {len(vocab['properties'])} canonical properties\n" + "-" * 74)
for t in sorted(vocab["properties"], key=lambda x: -x.get("support", 0)):
    al = f"  aka {t['aliases']}" if t.get("aliases") else ""
    print(f"  {t['id']}  {t['canonical_label']:<26} subPropertyOf {str(t.get('parent')):<16} "
          f"support={t.get('support')} | {_sig(t.get('domain'), t.get('range'))}{al}")

print("\nVERIFY ext vs oie_relations (every relation instance represented)\n" + "-" * 74)
ok = True
for aid in ARTICLE_ORDER:
    ext = RESULTS_DIR / aid / f"e2e/nary_graph{_osfx}_extended.json"
    if not ext.exists():
        print(f"  {aid:<8} (no ext file)"); continue
    oie = json.loads((RESULTS_DIR / aid / f"e2e/nary_graph{_isfx}_typed_consolidated.json").read_text())
    x   = json.loads(ext.read_text())["relations"]
    # every entity pair from oie must appear in some record. Compare UNORDERED pairs, since an inverse
    # reuse legitimately swaps subject/object — the pair is preserved, only its orientation changes.
    orig_pairs = {frozenset((r["subject_id"], r["object_id"])) for r in oie["relations"]}
    got_pairs  = {frozenset((r["subject_id"], r["object_id"])) for r in x}
    nvoc = sum(1 for r in x if r.get("property_source") == "vocab")
    nflip = sum(1 for r in x if "inverse" in (r.get("phrase_directions") or {}).values())
    covered = orig_pairs <= got_pairs
    ok &= covered
    print(f"  {aid:<8} {len(oie['relations']):>4} instances -> {len(x):>4} triples "
          f"({nvoc} via vocab, {nflip} inverse-reused)   {'OK' if covered else 'FAIL: pair(s) dropped'}")
print("-" * 74); print("consistent" if ok else "inconsistencies")

PROPERTY VOCABULARY — 33 canonical properties
--------------------------------------------------------------------------
  P0016  involves                   subPropertyOf None             support=215 | SpaceMission/EducationEvent/TestFlight/MoonLanding/Recovery/BatonPassing/Launch/Landing/SportsEvent/ReturnToFlight/Delay/AlternateMission/Adaptation/Presence/Living/Threat/Construction → Date/Vehicle/SpaceMission/SpaceAgency/Astronaut/Orbit/CelestialBody/Resource/Duration/MilitaryBranch/LunarLander/Infrastructure/Event/Corporation/Procedure/BusinessMagnate/Country/Place/SpaceProgram/Profession/Role/NASAOfficial/LaunchPad/Airport/CivicStructure/Satellite/Group/HistoricalPeriod/SportsEvent/NewsMediaOrganization/EducationalOccupationalProgram/MilitaryRank/Title/Symbol/SpaceStation
  P0033  plans                      subPropertyOf None             support=25 | SpaceAgency → SpaceMission/TestFlight/Launch/Construction/Presence/SportsEvent/AlternateMission/Threat/MoonLanding/Landing/Living/Ret

*The **global re-consolidation** and **property-hierarchy** passes follow below. They operate on `nary_relation_vocabulary<_osfx>.json` and the per-article `nary_graph<_osfx>_extended.json` files, and write only NEW `*_consolidated` files — the incremental outputs above are left untouched, so before/after is always comparable.*

# ─────────────────────────── GLOBAL RE-CONSOLIDATION ───────────────────────────
## Everything below is the global-reconsolidation block

The property-side twin of 6d's re-consolidation. Run the pipeline above (through **Build** and
**Report + verify**) to produce the raw incremental property vocabulary
(`nary_relation_vocabulary<_osfx>.json`), then run this block to clean it globally — merging
synonymous properties, generalizing several specifics into one newly minted term, or linking a
specific property under a broader one as `subPropertyOf`.

Every knob it uses — `CONSOLIDATE`, `SUBPROP_MODE`, `ALLOW_GENERALIZE`, `TWO_CALL_CONSOLIDATION`,
`GUARD_SCHEMA_PARENT`, `CONSOLIDATE_TOP_K`, `CONSOLIDATE_INCLUDE_SCHEMA`, `RERANK_CONSOLIDATE(_FROM)`
— comes from the experiment config, not from a cell here. Setting `consolidate: false` (the
**no-consolidation** ablation) skips the LLM judgement but still writes both output files, identical
to the raw ones, so everything downstream still runs.

**With `INLINE_VOCAB_CANDIDATES = False` this is not optional in practice.** The inline stage cannot
reuse an earlier mint (it only ever sees schema.org), so it re-mints synonyms of terms it already
created; this pass is what collapses them. Judge that configuration on the vocabulary size **after**
consolidation, not on the inline mint rate.

This block writes only **NEW files** — every incremental output is left untouched, so you can always
compare before/after.

**Candidate pool — `CONSOLIDATE_INCLUDE_SCHEMA`.** Every vocabulary property is embedded and
shortlisted against *one ranked pool* (`CONSOLIDATE_TOP_K` nearest neighbours each); those neighbours
are the only merge candidates the judge may use.

* **`False` (vocab-only)** — the pool is the other minted properties. The extension is canonicalized
  against itself and a group's survivor is always another `P…` entry. This is the right setting when
  the inline stage already ran schema.org-only (`INLINE_VOCAB_CANDIDATES = False`).
* **`True` (mixed)** — the pool also holds the full schema.org property pool, so a minted property that
  merely restates a standard one can be **folded into schema.org** (the extension entry disappears, its
  relation instances re-labelled) or made a **`subPropertyOf` it**. schema.org properties are fixed —
  survivor / parent only, never merged away, so only `P…` ids appear in `merge_ids`.

Direction rules either way: the prompt shows each candidate's `domain → range` signature and
`inverse_of`, and forbids grouping a relation with its inverse.

**Reranking — `RERANK_CONSOLIDATE`.** When on, e5 fetches `RERANK_CONSOLIDATE_FROM` neighbours and a
cross-encoder picks the `CONSOLIDATE_TOP_K` actually shown — the same two-stage retrieval Hook A uses,
but at its own depth (this pool is the vocabulary, orders smaller than schema.org, so it needs less). The score
printed beside each candidate stays the embedding cosine.

**Two guards on the schema.org side.** `SCHEMA_EXCLUDE_LABELS` drops schema.org's metamodel properties
(`domainIncludes`, `rangeIncludes`, `supersededBy`, `inverseOf`) from the pool — they describe the
vocabulary itself, not any relation an article states. And when a schema.org keep shares its **label**
with one of its children, that is a duplicate rather than a specialisation: the child is merged into
schema.org instead of nested under it, which would otherwise make the property its own `subPropertyOf`
(`parent` holds a bare label, and the vocab side wins the lookup) and drop it from the hierarchy. A final
sweep nulls any self-parent that slips through and lists it in the report as `self_parent_fixed`.

**Prompt layout.** One interleaved listing: each vocabulary property's line, then its candidates
indented under it as `->` lines, each carrying its own score, source, super-property, observed
`domain → range` signature, `inverse_of` (schema.org) and definition (truncated to
`CANDIDATE_DEF_CHARS`). Nothing is referenced by bare id, so the judge reads one property and its
options together — signatures sit side by side, which is exactly what the direction rule needs.

Because the LLM only sees each property's shortlist (not the full cross-product), the pass scales the
way the incremental build does — the prompt grows as `N × (1 + TOP_K)` lines.

The block, in order:
1. **Config + prompts** — output paths, `load_consolidated`, and the three prompts: `CONSOLIDATE_ALL_PROMPT` (single call) or `MERGE_PROMPT` + `REPARENT_PROMPT` (two calls).
2. **`consolidate_vocab`** — mixed-pool shortlist inlined per property + one or two LLM calls → `nary_relation_vocabulary<_osfx>_consolidated.json` + `nary_relation_consolidation_report<_osfx>.json` (with the full `neighbour_shortlist`, each candidate tagged `vocab` / `schema.org`, and an `in_shortlist` audit flag per merge/re-parent).
3. **`refresh_ext_after_consolidation`** — no LLM; writes `nary_graph_extended_consolidated.json` per article + a changes-only `relation_consolidation_ext_changes.json`.
4. **`build_property_hierarchy`** — indented schema.org + extension property tree, into `property_hierarchy_ext<_osfx>.txt` (raw) and `property_hierarchy_ext<_osfx>_consolidated.txt` (consolidated).

In [20]:
# Global consolidation prompts. Everything that decides WHAT this pass may do comes from the
# experiment config — do NOT reassign the knobs here, or the cell would override the run's record.
#
#   CONSOLIDATE             False -> skip the LLM judgement entirely (the `no-consolidation`
#                           ablation). The *_consolidated files are still written, identical to the
#                           raw ones, so everything downstream runs and the metrics show a real 0.
#   SUBPROP_MODE            how a SPECIFIC property relates to a broader one already in the pool:
#                             "off"          only merge exact synonyms
#                             "subproperty"  KEEP the specific one, set its parent (subPropertyOf)
#                                            to the broader one — an is-a-kind-of hierarchy
#                             "merge"        COLLAPSE the specific one into the broader parent
#   ALLOW_GENERALIZE        the third group kind: several specifics that are ONE relation seen at
#                           different levels of detail collapse into a NEWLY MINTED general property
#                           the judge names and defines. Off = only SYNONYM / SUBPROPERTY, so the two
#                           settings are directly comparable under separate experiments.
#   TWO_CALL_CONSOLIDATION  True: pass 1 removes REDUNDANCY (synonym + generalize) with no hierarchy
#                           option at all, then pass 2 builds the HIERARCHY over the survivors, with
#                           shortlists retrieved FRESH from the post-merge vocabulary. That removes
#                           the escape hatch — pass 1 cannot answer "subproperty" to dodge a merge —
#                           and puts the two decisions in the right order, since re-parenting should
#                           see the merged vocabulary rather than one its own answer is about to
#                           change. False: the original single call over CONSOLIDATE_ALL_PROMPT.
#                           Costs one extra LLM call + one extra embed/rerank pass per RUN (not per
#                           article).
#   GUARD_SCHEMA_PARENT     refuse any re-parent that would move a property OFF a schema.org parent
#                           onto a minted one — that lowers grounding, the opposite of this pass's
#                           job. Blocked moves are listed in the report as `blocked_degrounding`
#                           rather than silently applied.
#   CONSOLIDATE_TOP_K       nearest-neighbour shortlist size per property (retrieve-then-judge)
#   CONSOLIDATE_INCLUDE_SCHEMA  True: MIXED pool (vocab u the whole schema.org property pool), so a
#                           minted property that duplicates a standard one can be folded INTO
#                           schema.org or parented under it; schema.org properties are fixed —
#                           survivor/parent only, never merged away. False: VOCAB-ONLY.
assert SUBPROP_MODE in ("off", "subproperty", "merge"), "subprop_mode must be 'off', 'subproperty' or 'merge'"

SCHEMA_ID_PREFIX = "schema:"    # how a schema.org candidate is identified to the LLM (e.g. schema:author)
CANDIDATE_DEF_CHARS = 200       # per-candidate definition budget (candidates are inlined under each property)
# schema.org METAMODEL properties describe the vocabulary itself (which class a property belongs to,
# what supersedes what) rather than any relation an article states, so they are dropped from the
# candidate pool — the property-side twin of 6d's SCHEMA_EXCLUDE_LABELS. Empty the set to allow them back.
SCHEMA_EXCLUDE_LABELS = {"domainIncludes", "rangeIncludes", "supersededBy", "inverseOf"}

# Consolidation writes to SEPARATE files — the raw nary_relation_vocabulary<_osfx>.json stays INTACT.
CONSOLIDATED_VOCAB_PATH   = RESULTS_DIR / f"_global/e2e/nary_relation_vocabulary{_osfx}_consolidated.json"
CONSOLIDATION_REPORT_PATH = RESULTS_DIR / f"_global/e2e/nary_relation_consolidation_report{_osfx}.json"
def load_consolidated():
    """The consolidated vocab if it exists, else fall back to the raw incremental one."""
    return json.loads(CONSOLIDATED_VOCAB_PATH.read_text()) if CONSOLIDATED_VOCAB_PATH.exists() else load_vocab()

CONSOLIDATE_ALL_PROMPT = """You are consolidating a vocabulary of relation properties. Each entry is a directed relation subject --property--> object. You are given every vocabulary property (id, label, parent, support, domain -> range signature, definition) and, indented under it, its CANDIDATES — the only entries it may be grouped with.

Aim for the SMALLEST vocabulary that still expresses every relation. Redundancy is the failure to avoid: where two entries could reasonably be one, make them one.

ID NAMESPACES
- `P...` — vocabulary properties minted by this pipeline. Only these may appear in merge_ids.
- `schema:x` — standard schema.org properties. They may appear as keep_id ONLY, never in merge_ids, and are never removed. Prefer a schema.org survivor whenever one fits: an extension property that restates a standard one is redundant.

THREE KINDS OF GROUP

SYNONYM — entries denoting the same relation in the same direction, differing only in wording. keep_id = the survivor (a `schema:x` id whenever one fits), merge_ids = the rest.

GENERALIZE — two or more entries that are one relation described at different levels of detail, where NO existing entry names the general relation. You name it: merge_ids = the specifics, new_label = a camelCase name for the general relation, new_definition = one sentence, new_parent = an existing broader property label or null. No keep_id. If schema.org or another vocabulary entry ALREADY names the general relation, use SYNONYM into it instead — never mint a name for something that already exists.

SUBPROPERTY — a genuinely narrower relation kept under a broader one. keep_id = the broader parent, merge_ids = the children. This is the rarest kind, and it must pass the test below.

THE SUBPROPERTY TEST
A child is a real subproperty only if it constrains the RELATION itself. Merge or generalize it instead when the only thing making it "specific" is:
(a) the type of its object — one relation split into variants named after the kind of value it points at;
(b) the modality of the assertion — stated, reported, expected, forecast, observed, actual: these say how a fact is known, not which relation holds;
(c) the subject or setting it was seen in — a variant whose label names the context rather than the relation.
All three distinctions belong on the node that carries the fact, not in the property name. If you cannot say what a child constrains beyond (a), (b) and (c), it is not a subproperty.

RULES
- Group only entries listed among a property's own candidates, or that list it among theirs. Never invent an id.
- Same direction only: check domain -> range on both sides, and never group a relation with its inverse (`inverse_of` marks the schema.org properties that have one).
- A vocabulary entry sharing its LABEL with a schema.org candidate is a duplicate of it, not a specialisation: SYNONYM with the `schema:x` id as keep_id.
- Never move a property off a schema.org parent onto a vocabulary one.
- Each vocabulary property belongs to at most one group. Give one short reason per group.
- A new_label names the relation alone — no object type, no modality, no subject baked into it — and must still read correctly for arguments of completely different types.

Return ONLY valid JSON — no prose, no code fences. <angle brackets> show the shape, not literal values:
{"groups": [{"kind": "synonym", "keep_id": "<schema:x or P id>", "merge_ids": ["<P id>"], "reason": "<one line>"}, {"kind": "generalize", "new_label": "<camelCase>", "new_definition": "<one sentence>", "new_parent": "<label or null>", "merge_ids": ["<P id>", "<P id>"], "reason": "<one line>"}, {"kind": "subproperty", "keep_id": "<parent id>", "merge_ids": ["<child id>"], "reason": "<one line>"}]}"""

# ── PASS 1 of TWO_CALL_CONSOLIDATION: redundancy only. It is given no hierarchy option and no
# mention of one — the second pass is deliberately invisible here, so nothing primes the model to
# reach for a parent/child answer. What keeps it from over-merging instead is the ATTRIBUTE test
# below, the hinge this prompt shares with REPARENT_PROMPT: a label naming which attribute is
# measured is a distinct relation (pass 2's job), not a wording variant.
# Rules the code already enforces are NOT restated here — see _keep_ref, _split_label_collisions,
# _normalize_generalize, _only and the merged_ids guard.
MERGE_PROMPT = """You are consolidating a vocabulary of relation properties, each a directed relation subject --property--> object. You are given every property (id, label, parent, support, domain -> range signature, definition) and under each one its CANDIDATES: the only entries it may be grouped with.

Aim for the smallest vocabulary that still expresses every relation, without forcing a group: an entry that genuinely matches nothing stays on its own.

Find sets of entries that all express the same relation and collapse each set onto one name, given in whichever of these two ways fits:
- keep_id: the id of an entry that already names the relation. It may be one of the set, or any other property listed among their candidates, including a schema.org one. merge_ids = every entry of the set except keep_id itself.
- new_label: a camelCase name you supply, when nothing names the relation yet and the set folds into one more general term. List every entry of the set in merge_ids, and add new_definition (one sentence) and new_parent (a broader property label, or null).
Never supply a new_label for a name that already exists: use keep_id instead.

Entries express the same relation when the only thing separating them is:
(a) the value their object takes: the same statement at a different granularity or in a different value type. For example, `priceInEuros` and `priceInUsd` are both `price`.
(b) the modality of the assertion: stated, reported, expected, forecast, observed, actual. For example, `forecastRevenue` and `actualRevenue` are both `revenue`.
(c) the subject or setting they were seen in. For example, `matchAttendance` and `concertAttendance` are both `attendance`.
All three distinctions belong on the node that carries the fact, not in the property name.
They express different relations when a label names which attribute of the subject is measured: an area, a count, a rate, a percentage, a sum of money. That is part of the relation's identity, not just the type of its object. For example, `population` and `area` express different relations.
The test: if both labels could describe the same subject --> object pair, they express the same relation; if not, leave both ungrouped.

Rules:
- `P...` ids are vocabulary properties minted by this pipeline; only these may appear in merge_ids.
- `schema:x` ids are standard schema.org properties; they are fixed, so they may appear only as keep_id.
- Prefer a `schema:x` keep_id whenever one fits: an extension property that restates a standard one is redundant.
- Judge direction from each side's domain -> range, and never group a relation with its inverse.
- A new_label names the relation alone, with no object type, modality or subject baked into it, and must read correctly for arguments of completely different types.

Return only valid JSON, no prose and no code fences:
{"groups": [{"keep_id": "<schema:x or P id>", "merge_ids": ["<P id>"], "reason": "<one line>"}, {"new_label": "<camelCase>", "new_definition": "<one sentence>", "new_parent": "<label or null>", "merge_ids": ["<P id>", "<P id>"], "reason": "<one line>"}]}"""

# ── PASS 2 of TWO_CALL_CONSOLIDATION: hierarchy only, over the survivors of pass 1. Nothing can be
# removed here, so parenting costs no information. This prompt deliberately does NOT tell the model
# that leaving an entry unparented is preferable: that framing was calibrated against the old
# over-parenting, which was a symptom of merge avoidance pass 1 now prevents, and it suppressed
# obvious links (hasAlert under hasRisk) in the first two-call run.
REPARENT_PROMPT = """You are adding hierarchy to a vocabulary of relation properties, each a directed relation subject --property--> object. Redundancy has already been removed: every entry is distinct, and nothing may be merged, renamed or deleted — a child only gains a parent.

You are given every property (id, label, current parent, support, domain -> range signature, definition) and, indented under it, its CANDIDATES — the only entries it may be linked to.

Return SUBPROPERTY groups: parent_id = the broader parent (a `P...` id or a `schema:x` property), children_ids = the narrower children to attach under it as subPropertyOf.

A child is a real subproperty when every instance of the child is still a valid instance of the parent, in the same direction.

Rules:
- Prefer a `schema:x` parent: it grounds the extension in the standard vocabulary.
- Each property gets at most one parent, so propose each child once.
- Judge direction from each side's domain -> range. Never parent a relation under its inverse, or under one of its own descendants.

Return only valid JSON — no prose, no code fences:
{"groups": [{"kind": "subproperty", "parent_id": "<parent id>", "children_ids": ["<child id>"], "reason": "<one line>"}]}"""

print(f"consolidation prompts loaded | enabled={CONSOLIDATE} | "
      f"calls={2 if TWO_CALL_CONSOLIDATION else 1} | "
      f"SUBPROP_MODE={SUBPROP_MODE!r} | generalize={ALLOW_GENERALIZE} | "
      f"schema-parent guard={GUARD_SCHEMA_PARENT} | "
      f"mixed pool (schema.org candidates): {CONSOLIDATE_INCLUDE_SCHEMA} | consolidated -> {CONSOLIDATED_VOCAB_PATH.name}")

consolidation prompts loaded | enabled=True | calls=2 | SUBPROP_MODE='subproperty' | generalize=True | schema-parent guard=True | mixed pool (schema.org candidates): True | consolidated -> nary_relation_vocabulary_dedup_main_test_consolidated.json


In [21]:
def _consolidation_disabled():
    """CONSOLIDATE=False: copy the raw vocabulary to the consolidated path and write an empty
    report, so every consumer downstream is unchanged and the run is still fully described."""
    v = load_vocab()
    CONSOLIDATED_VOCAB_PATH.write_text(json.dumps(v, indent=2, ensure_ascii=False))
    rep = {"timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "consolidation": "DISABLED",
           "experiment": EXPERIMENT, "subprop_mode": SUBPROP_MODE,
           "source_vocab": VOCAB_PATH.name, "consolidated_vocab": CONSOLIDATED_VOCAB_PATH.name,
           "vocab_size_before": len(v["properties"]), "vocab_size_after": len(v["properties"]),
           "n_merges": 0, "n_props_removed": 0, "n_reparented": 0, "n_merged_into_schema": 0,
           "n_reparented_under_schema": 0, "n_generalize_groups": 0, "n_generalized": 0,
           "merges": [], "reparents": [], "generalizations": [], "blocked_degrounding": [],
           "parent_conflicts": [], "keep_overrides": [],
           "neighbour_shortlist": [], "neighbour_shortlist_reparent": [], "self_parent_fixed": []}
    CONSOLIDATION_REPORT_PATH.write_text(json.dumps(rep, indent=2, ensure_ascii=False))
    print(f"consolidation DISABLED for experiment {EXPERIMENT!r} — copied the raw vocabulary "
          f"({len(v['properties'])} properties) to {CONSOLIDATED_VOCAB_PATH.name} unchanged")
    return rep


# Retrieve-then-judge: every vocab property is shortlisted against the other vocab properties, plus the
# full schema.org property pool when CONSOLIDATE_INCLUDE_SCHEMA is on (then the judge sees standard and
# sibling extension properties side by side as merge candidates). With RERANK_CONSOLIDATE on, e5 fetches
# RERANK_CONSOLIDATE_FROM neighbours and a cross-encoder picks the CONSOLIDATE_TOP_K shown to the
# judge — the same two-stage retrieval Hook A uses, at its own depth. The score printed for each
# candidate stays the EMBEDDING cosine.
def _candidate_line(src, obj, score):
    """One rendered candidate, inlined under the property it was retrieved for — carries everything the
    judge needs (id, label, super-property, domain -> range signature, inverse, definition), so no
    separate lookup block is needed."""
    if src == "schema.org":
        desc = (obj.get("enriched_comment") or obj.get("comment") or "").replace("\n", " ").strip()
        inv  = f' | inverse_of: {obj.get("inverse_of")}' if obj.get("inverse_of") else ""
        return (f'    -> {SCHEMA_ID_PREFIX}{obj["label"]} ({score}) | schema.org'
                f' | parent: {_schema_super(obj["label"])} | {obj.get("signature", "")}{inv}'
                f' | def: {desc[:CANDIDATE_DEF_CHARS]}')
    desc = (obj.get("definition") or "").replace("\n", " ").strip()
    return (f'    -> {obj["id"]} ({score}) | vocab | label: {obj["canonical_label"]}'
            f' | parent: {obj.get("parent")} | support: {obj.get("support", 0)}'
            f' | {_sig(obj.get("domain"), obj.get("range"))}'
            f' | def: {desc[:CANDIDATE_DEF_CHARS]}')

def _consolidation_shortlist(ents_sorted):
    """Top-CONSOLIDATE_TOP_K nearest neighbours of each vocab property over (vocab \\ self) u schema.org.
    Returns (cand_lines, shortlist_records, adj): `cand_lines` maps a vocab id to its rendered candidate
    lines (spliced under that property's line in the prompt), `adj` maps it to the set of candidate ids
    the embedder proposed (used for the audit), `shortlist_records` is the same thing for the report."""
    ids  = [t["id"] for t in ents_sorted]
    labs = [t["canonical_label"] for t in ents_sorted]
    V    = embed([vocab_embedding_text(t) for t in ents_sorted], DP)
    meta = [("vocab", ids[i], labs[i], ents_sorted[i]) for i in range(len(ents_sorted))]
    D    = V
    if CONSOLIDATE_INCLUDE_SCHEMA:
        keep = [i for i, p in enumerate(schema_pool) if p["label"] not in SCHEMA_EXCLUDE_LABELS]
        D    = np.vstack([V, schema_emb[keep]])
        meta = meta + [("schema.org", SCHEMA_ID_PREFIX + schema_pool[i]["label"],
                        schema_pool[i]["label"], schema_pool[i]) for i in keep]
    sims = V @ D.T
    for i in range(len(ents_sorted)):
        sims[i, i] = -1.0                                   # a property is never its own candidate
    k = min(CONSOLIDATE_TOP_K, D.shape[0] - 1)

    def _cand_doc(m):
        """"label: definition" for the cross-encoder — vocab entries and schema.org entries alike."""
        obj = m[3]
        d = obj.get("definition") or obj.get("enriched_comment") or obj.get("comment") or ""
        return f"{m[2]}: {d[:400]}"

    fetch = min(max(k, RERANK_CONSOLIDATE_FROM), D.shape[0] - 1) if RERANK_CONSOLIDATE else k
    cand_lines, shortlist, adj = {}, [], {}
    for i in range(len(ents_sorted)):
        top = np.argsort(-sims[i])[:fetch]
        if RERANK_CONSOLIDATE and len(top) > k:      # cross-encode the shortlist, same as Hook A
            q  = vocab_embedding_text(ents_sorted[i])
            rs = np.asarray(_reranker().predict([(q, _cand_doc(meta[j])) for j in top],
                                                batch_size=64, show_progress_bar=False))
            top = top[np.argsort(-rs)][:k]
        # the score shown to the LLM stays the EMBEDDING cosine — a cross-encoder logit is on a
        # different scale and would read as a confidence it does not carry.
        cands = [(meta[j], round(float(sims[i][j]), 3)) for j in top]
        adj[ids[i]] = {m[1] for m, _ in cands}
        cand_lines[ids[i]] = [_candidate_line(m[0], m[3], s) for m, s in cands]
        shortlist.append({"id": ids[i], "label": labs[i],
                          "neighbours": [{"id": m[1], "label": m[2], "source": m[0], "score": s}
                                         for m, s in cands]})
    return cand_lines, shortlist, adj

def consolidate_vocab(vocab, write_report=True, verbose=True):
    """Global re-consolidation of the PROPERTY vocabulary in ONE Gemini call, judged against the MIXED
    candidate pool (vocab u schema.org). The model returns SYNONYM and SUBPROPERTY groups whose keep_id
    may be a vocab id OR a `schema:x` property; how SUBPROPERTY (is-a) groups are applied depends on
    SUBPROP_MODE:
      "off"         -> ignore them (only synonyms merge).
      "subproperty" -> KEEP each child, set its parent (subPropertyOf) to the broader property
                       (vocab OR schema.org).
      "merge"       -> fold each child into the broader parent (removed; label -> alias).
    SYNONYM groups merge onto the highest-support member, or onto the schema.org property when the LLM
    kept one (the minted property is then dropped from the extension entirely). Writes the consolidated
    vocab to CONSOLIDATED_VOCAB_PATH and a report to CONSOLIDATION_REPORT_PATH — the raw
    relation_vocabulary.json is NOT modified (pass load_vocab() as `vocab`)."""
    ents  = list(vocab["properties"])
    by_id = {t["id"]: t for t in ents}

    def _articles_of(t):
        return [p.get("article") for p in t.get("provenance", []) if p.get("article")]
    articles_before = sorted({a for t in ents for a in _articles_of(t)})

    shortlist, shortlist_r, _adj, n_offkind = [], [], {}, 0

    def _judge(items, system_prompt, gate, what):
        """ONE shortlist + ONE LLM call over `items`. Each property is followed by its own candidates
        (vocab and schema.org ranked together), so every id the judge may reach for is defined right
        where it is offered. Returns (groups, shortlist_records, adjacency)."""
        if len(items) < 2:
            return [], [], {}
        lines = [f'- id: {t["id"]} | label: {t["canonical_label"]} | parent: {t.get("parent")} '
                 f'| support: {t.get("support", 0)} | {_sig(t.get("domain"), t.get("range"))} '
                 f'| def: {t.get("definition", "")}'
                 + (f' | aka: {", ".join(t["aliases"])}' if t.get("aliases") else "")
                 for t in items]
        cand_lines, sl, adj = _consolidation_shortlist(items)
        blocks = [ln + "\n" + "\n".join(cand_lines[t["id"]]) for t, ln in zip(items, lines)]
        msg = ("VOCABULARY — each property, then its retrieved CANDIDATES (`->` lines: the only entries "
               "it may be grouped with):\n\n" + "\n\n".join(blocks))
        try:
            res = parse_json(gemini_json(system_prompt + gate, msg))
            _groups = (res.get("groups", res) if isinstance(res, dict) else res) or []
            # A TRUNCATED reply comes back through json_repair structurally wrong — a bare list
            # where a group dict was expected — and the g.get("kind"/"keep_id"/...) calls
            # downstream then raise AttributeError and kill the STAGE, not the call. That is how
            # qwen3-32b-think lost 6d fifty articles in on 2026-08-24: it looped on one
            # consolidation call, truncated at 26k output tokens, and json_repair salvaged 61k
            # chars into the wrong shape. A non-dict group carries no decision, so drop it and let
            # the rest of the pass stand rather than losing the run.
            _malformed = [g for g in _groups if not isinstance(g, dict)]
            if _malformed:
                print(f"  [{what}] dropped {len(_malformed)} malformed group(s) from the reply "
                      f"(saw: {sorted({type(g).__name__ for g in _malformed})}) — the call was "
                      f"probably truncated; the surviving groups are still applied")
                _groups = [g for g in _groups if isinstance(g, dict)]
            return _groups, sl, adj
        except Exception as ex:
            print(f"[WARN] {what} call failed ({type(ex).__name__}: {ex})")
            return [], sl, adj

    def _only(groups, want_subprop, label):
        """Keep just the kinds this pass is allowed to return; count what it dropped. The separation
        between the two passes is enforced here, not merely requested in the prompt."""
        nonlocal n_offkind
        keep = [g for g in groups
                if (g.get("kind") or "").lower().startswith("subprop") == want_subprop]
        if len(keep) != len(groups):
            print(f"  [note] dropped {len(groups) - len(keep)} off-kind group(s) from the {label} pass")
            n_offkind += len(groups) - len(keep)
        return keep

    def _reparent_fields(groups):
        """REPARENT_PROMPT names its fields parent_id / children_ids, which is what pass 2 actually
        does: nothing is merged there, a child only gains a parent. Map them onto the keep_id /
        merge_ids the rest of the pipeline uses, and default the kind, since pass 2 returns only
        subproperty groups."""
        out = []
        for g in groups:
            g = dict(g)
            if "parent_id" in g:
                g.setdefault("keep_id", g.pop("parent_id"))
            if "children_ids" in g:
                g.setdefault("merge_ids", g.pop("children_ids"))
            g.setdefault("kind", "subproperty")
            out.append(g)
        return out

    def _one_parent(groups):
        """One parent per property. The judge does emit competing groups — a child offered to two or
        three parents — and nothing in the data model stops a later assignment from silently
        overwriting an earlier one, so the conflict is resolved HERE, before anything is applied.
        Among the parents proposed for one child, prefer a parent that is itself being parented in the
        same answer: it is the NEARER ancestor, and a subproperty belongs under its nearest subsuming
        property rather than a distant one. Remaining ties go to the first proposal. Every rejected
        proposal is recorded in the report instead of vanishing."""
        deeper = {c for g in groups for c in (g.get("merge_ids") or [])}   # parents that have a parent
        chosen = {}
        for gi, g in enumerate(groups):
            score = (0 if g.get("keep_id") in deeper else 1, gi)
            for cid in dict.fromkeys(g.get("merge_ids") or []):
                if cid not in chosen or score < chosen[cid][0]:
                    chosen[cid] = (score, gi)
        out = []
        for gi, g in enumerate(groups):
            ids  = list(dict.fromkeys(g.get("merge_ids") or []))
            keep = [c for c in ids if chosen.get(c, (None, gi))[1] == gi]
            for c in (c for c in ids if c not in keep):
                parent_conflicts.append({"id": c, "label": by_id[c]["canonical_label"] if c in by_id else c,
                                         "kept_parent": groups[chosen[c][1]].get("keep_id"),
                                         "rejected_parent": g.get("keep_id")})
            if keep:
                out.append({**g, "merge_ids": keep})
        return out

    def _keep_ref(keep):
        """Resolve a keep_id to ("vocab", entry) or ("schema.org", label); (None, None) if unknown.
        A bare schema.org label (no `schema:` prefix) is tolerated."""
        if not isinstance(keep, str):
            return None, None
        if keep in by_id:
            return "vocab", by_id[keep]
        lbl = keep[len(SCHEMA_ID_PREFIX):] if keep.startswith(SCHEMA_ID_PREFIX) else keep
        if lbl in SCHEMA_LABELS and lbl not in SCHEMA_EXCLUDE_LABELS:   # excluded ones are never valid keeps
            return "schema.org", lbl
        return None, None

    def _split_label_collisions(groups):
        """A schema.org keep that shares its LABEL with one of its children is not a hierarchy — it is
        the same name twice. `parent` stores a bare label, so attaching the child there would make it
        its OWN parent (self-loop: _find_vocab_by_label resolves the label back to the child, and the
        entry drops out of the hierarchy). An identical label means equivalence, so those children are
        rewritten into a SYNONYM merge into schema.org; any remaining children keep the original kind."""
        out = []
        for g in groups:
            ksrc, kobj = _keep_ref(g.get("keep_id"))
            ids  = list(dict.fromkeys(g.get("merge_ids") or []))
            same = [i for i in ids if ksrc == "schema.org" and i in by_id
                    and by_id[i]["canonical_label"] == kobj]
            if not same:
                out.append(g); continue
            note = " [same label as the schema.org entry -> merged, not nested under it]"
            out.append({**g, "kind": "synonym", "merge_ids": same,
                        "reason": (g.get("reason") or "") + note})
            rest = [i for i in ids if i not in same]
            if rest:
                out.append({**g, "merge_ids": rest})
        return out
    def _derive_kind(groups):
        """MERGE_PROMPT no longer asks for a "kind": a group either points at an existing survivor
        (keep_id) or supplies a name that does not exist yet (new_label). Recover the kind from that
        shape, so _normalize_generalize / _apply below are unchanged. An explicit kind, if the judge
        volunteers one anyway, is left alone so _only can still filter it out."""
        out = []
        for g in groups:
            if (g.get("kind") or "").strip():
                out.append(g); continue
            mints = bool((g.get("new_label") or "").strip()) and not g.get("keep_id")
            out.append({**g, "kind": "generalize" if mints else "synonym"})
        return out

    def _normalize_generalize(groups):
        """A GENERALIZE names a general relation that does NOT exist yet. When the label it proposes
        already names something — one of its own children, a schema.org property, or another vocab
        entry — the group is really a SYNONYM into that term, so rewrite it instead of dropping it and
        the collapse still happens. Drops the kind entirely when ALLOW_GENERALIZE is off."""
        out = []
        for g in groups:
            if not (g.get("kind") or "").lower().startswith("general"):
                out.append(g); continue
            ids   = [i for i in dict.fromkeys(g.get("merge_ids") or []) if i in by_id]
            label = (g.get("new_label") or "").strip()
            if not ALLOW_GENERALIZE or len(ids) < 2 or not label:
                continue
            own      = next((by_id[i] for i in ids if by_id[i]["canonical_label"] == label), None)
            existing = _find_vocab_by_label(vocab, label)
            if own is not None:                                   # the "new" name is one of its children
                out.append({**g, "kind": "synonym", "keep_id": own["id"],
                            "merge_ids": [i for i in ids if i != own["id"]]})
            elif label in SCHEMA_LABELS and label not in SCHEMA_EXCLUDE_LABELS:
                out.append({**g, "kind": "synonym", "keep_id": SCHEMA_ID_PREFIX + label,
                            "merge_ids": ids})                    # schema.org already names it
            elif existing is not None:
                out.append({**g, "kind": "synonym", "keep_id": existing["id"],
                            "merge_ids": [i for i in ids if i != existing["id"]]})
            else:
                out.append({**g, "merge_ids": ids})
        return out

    merges, reparents, merged_ids = [], [], set()
    generalizations, blocked_degrounding = [], []
    parent_conflicts, keep_overrides, assigned_parent = [], [], {}
    label_remap = {}          # merged-away label -> survivor label (to repair `parent` pointers)
    def _apply(groups):
        """Apply one pass's groups to the vocabulary. Called once per LLM call, so the
        accumulators below span both passes."""
        for g in groups:
            kind = (g.get("kind") or "synonym").lower()
            is_subprop = kind.startswith("subprop")

            # GENERALIZE: the specifics collapse into ONE general property that does not exist yet, so it
            # is MINTED here. Handled ahead of the keep_id dispatch because the survivor has no id to
            # resolve — _normalize_generalize has already rewritten any group whose name does exist.
            if kind.startswith("general"):
                children = [by_id[i] for i in dict.fromkeys(g.get("merge_ids") or [])
                            if i in by_id and i not in merged_ids]
                if len(children) < 2:
                    continue
                parent = (g.get("new_parent") or "").strip() or None
                if parent is not None and not (parent in SCHEMA_LABELS or _is_vocab_label(vocab, parent)):
                    parent = None                                     # unknown parent -> no super-property
                _ord = {a: i for i, a in enumerate(globals().get("ARTICLE_ORDER") or [])}
                ent = {"id": f"P{vocab['next_id']:04d}", "canonical_label": g["new_label"].strip(),
                       "definition": (g.get("new_definition") or "").strip(), "parent": parent,
                       "domain": sorted({d for c in children for d in (c.get("domain") or [])}),
                       "range":  sorted({r for c in children for r in (c.get("range")  or [])}),
                       "aliases": [], "support": 0,
                       "first_seen": min(children,
                                         key=lambda c: _ord.get(c.get("first_seen"), 10 ** 6)).get("first_seen"),
                       "provenance": []}
                rec = {"kind": "generalize", "action": "generalize", "new_id": ent["id"],
                       "new_label": ent["canonical_label"], "definition": ent["definition"],
                       "parent": parent, "reason": g.get("reason"), "merged": [],
                       "articles": sorted({a for c in children for a in _articles_of(c)})}
                for c in children:                                    # the specifics fold into the new term
                    for lbl in [c["canonical_label"]] + c.get("aliases", []):
                        if lbl != ent["canonical_label"] and lbl not in ent["aliases"]:
                            ent["aliases"].append(lbl)
                    ent["support"] += c.get("support", 0)
                    ent["provenance"].extend(c.get("provenance", []))
                    rec["merged"].append({"id": c["id"], "label": c["canonical_label"],
                                          "support": c.get("support", 0), "articles": _articles_of(c)})
                    merged_ids.add(c["id"])
                    label_remap[c["canonical_label"]] = ent["canonical_label"]
                vocab["properties"].append(ent); vocab["next_id"] += 1
                rec["resulting_support"] = ent["support"]; rec["resulting_aliases"] = list(ent["aliases"])
                generalizations.append(rec)
                continue

            ksrc, kobj = _keep_ref(g.get("keep_id"))
            if ksrc is None or (ksrc == "vocab" and kobj["id"] in merged_ids):
                continue
            # Only VOCAB properties can be merged away — a schema.org id in merge_ids is ignored by design.
            children = [by_id[i] for i in dict.fromkeys(g.get("merge_ids") or [])
                        if i in by_id and i not in merged_ids and not (ksrc == "vocab" and i == kobj["id"])]
            if not children:
                continue

            # SUBPROPERTY + subproperty mode: keep the children, just re-parent them onto the broader property
            if is_subprop and SUBPROP_MODE == "subproperty":
                parent_label = kobj["canonical_label"] if ksrc == "vocab" else kobj
                rec = {"kind": "subproperty", "action": "subproperty",
                       "parent_id": kobj["id"] if ksrc == "vocab" else SCHEMA_ID_PREFIX + kobj,
                       "parent_label": parent_label, "parent_source": ksrc,
                       "reason": g.get("reason"), "children": []}
                for child in children:
                    if child.get("parent") == parent_label:
                        continue                                      # already under this parent — no-op
                    # Re-parenting must never move a property OFF a schema.org parent onto a minted
                    # one: that lowers grounding, the opposite of this pass's job. Recorded, not applied.
                    if GUARD_SCHEMA_PARENT and ksrc == "vocab" and child.get("parent") in SCHEMA_LABELS:
                        blocked_degrounding.append({"id": child["id"], "label": child["canonical_label"],
                                                    "kept_parent": child.get("parent"),
                                                    "rejected_parent": parent_label})
                        continue
                    if child["id"] in assigned_parent:            # _one_parent should have caught it
                        parent_conflicts.append({"id": child["id"], "label": child["canonical_label"],
                                                 "kept_parent": assigned_parent[child["id"]],
                                                 "rejected_parent": parent_label})
                        continue
                    rec["children"].append({"id": child["id"], "label": child["canonical_label"],
                                            "old_parent": child.get("parent")})
                    child["parent"] = parent_label                    # keep the child; re-parent it
                    assigned_parent[child["id"]] = parent_label
                if rec["children"]:
                    reparents.append(rec)
                continue

            # otherwise MERGE: synonyms always; subproperty only when SUBPROP_MODE == "merge"
            if is_subprop and SUBPROP_MODE != "merge":
                continue
            if ksrc == "schema.org":
                survivor, members = None, children                    # folds INTO schema.org: nothing survives in vocab
                survivor_id, survivor_label = SCHEMA_ID_PREFIX + kobj, kobj
            elif is_subprop:
                survivor, members = kobj, children                    # broader parent survives
            else:
                members  = [kobj] + children
                # The judge's keep_id is HONOURED. Overriding it with the most-supported member is how
                # a modality-baked variant swallowed the general term it should have merged into
                # (forecastsTemperature, support 4, beating hasTemperature, support 1) — support counts
                # usage, which is a poor proxy for which label should be canonical. Where the two
                # disagree the choice is recorded, so the heuristic remains auditable.
                survivor = kobj
                _top = max(members, key=lambda t: (t.get("support", 0), -int(t["id"][1:])))
                if _top["id"] != kobj["id"]:
                    keep_overrides.append({"kept": kobj["canonical_label"],
                                           "kept_support": kobj.get("support", 0),
                                           "most_supported": _top["canonical_label"],
                                           "most_supported_support": _top.get("support", 0)})
            if survivor is not None:
                survivor_id, survivor_label = survivor["id"], survivor["canonical_label"]
            rec = {"kind": kind, "action": "merge", "survivor_id": survivor_id,
                   "survivor_label": survivor_label,
                   "survivor_source": "schema.org" if survivor is None else "vocab",
                   "reason": g.get("reason"), "merged": [],
                   "articles": sorted({a for t in members for a in _articles_of(t)})}
            for m in members:
                if survivor is not None and m["id"] == survivor["id"]:
                    continue
                if survivor is not None:                              # schema.org survivors hold no aliases/provenance
                    for lbl in [m["canonical_label"]] + m.get("aliases", []):
                        if lbl != survivor["canonical_label"] and lbl not in survivor["aliases"]:
                            survivor["aliases"].append(lbl)
                    survivor["support"] = survivor.get("support", 0) + m.get("support", 0)
                    survivor.setdefault("provenance", []).extend(m.get("provenance", []))
                rec["merged"].append({"id": m["id"], "label": m["canonical_label"],
                                      "support": m.get("support", 0), "articles": _articles_of(m)})
                merged_ids.add(m["id"])
                label_remap[m["canonical_label"]] = survivor_label
            if rec["merged"]:
                if survivor is not None:
                    rec["resulting_support"]  = survivor["support"]
                    rec["resulting_aliases"] = list(survivor["aliases"])
                merges.append(rec)

    # ── the judgement itself ───────────────────────────────────────────────────────────────
    ents_sorted = sorted(ents, key=lambda x: x["id"])
    if len(ents) >= 2 and TWO_CALL_CONSOLIDATION:
        # PASS 1 — redundancy. No hierarchy option is offered at all, so a near-tie cannot be
        # resolved by parenting instead of collapsing.
        gate_m = ("" if ALLOW_GENERALIZE else
                  "\nDo NOT return GENERALIZE groups; use only entries that already exist.")
        groups_m, shortlist, _adj = _judge(ents_sorted, MERGE_PROMPT, gate_m, "merge")
        _apply(_split_label_collisions(_normalize_generalize(_only(_derive_kind(groups_m), False, "merge"))))

        # PASS 2 — hierarchy over the SURVIVORS, with shortlists retrieved fresh from the post-merge
        # vocabulary: slots that merged-away entries were occupying now hold real candidates, and no
        # child can be parented under an entry pass 1 just deleted.
        if SUBPROP_MODE != "off":
            surv = sorted([t for t in vocab["properties"] if t["id"] not in merged_ids],
                          key=lambda x: x["id"])
            by_id = {t["id"]: t for t in surv}          # pass 2 may only reach surviving ids
            groups_r, shortlist_r, adj_r = _judge(surv, REPARENT_PROMPT, "", "re-parent")
            _apply(_one_parent(_only(_reparent_fields(groups_r), True, "re-parent")))
            for _k, _v in adj_r.items():                # both passes' proposals feed the audit
                _adj[_k] = set(_adj.get(_k, set())) | set(_v)
    elif len(ents) >= 2:
        # single-call path — both kinds in one answer (the original behaviour, kept for comparison)
        gate = ("\nSUBPROPERTY groups are enabled." if SUBPROP_MODE != "off" else
                "\nDo NOT return SUBPROPERTY groups; collapse broader/narrower pairs instead.")
        gate += ("" if ALLOW_GENERALIZE else
                 "\nDo NOT return GENERALIZE groups; use only entries that already exist.")
        groups, shortlist, _adj = _judge(ents_sorted, CONSOLIDATE_ALL_PROMPT, gate, "consolidation")
        _apply(_split_label_collisions(_normalize_generalize(groups)))

    if merged_ids:
        vocab["properties"] = [t for t in vocab["properties"] if t["id"] not in merged_ids]
    # A merged-away label can still be some surviving property's `parent` (subPropertyOf) — repoint it
    # at the survivor (following merge chains), so no super-property link dangles after consolidation.
    for t in vocab["properties"]:
        p, seen = t.get("parent"), set()
        while p in label_remap and p not in seen:
            seen.add(p); p = label_remap[p]
        if p != t.get("parent"):
            t["parent"] = p
    # Safety net: an entry must never be its own parent. A bare label can name both a vocab entry and a
    # schema.org one, and the vocab side wins the lookup — so a self-label parent silently orphans the
    # entry from the hierarchy. Drop it to no super-entry and record it in the report.
    self_parented = [t["id"] for t in vocab["properties"] if t.get("parent") == t["canonical_label"]]
    for t in vocab["properties"]:
        if t.get("parent") == t["canonical_label"]:
            t["parent"] = None

    # write the CONSOLIDATED vocabulary to its own file (raw relation_vocabulary.json stays untouched)
    _annotate_vocab_paths(vocab)                 # refresh derived `path`s (re-parenting/merges changed them)
    CONSOLIDATED_VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

    n_reparented  = sum(len(r["children"]) for r in reparents)
    n_into_schema = sum(len(r["merged"]) for r in merges if r["survivor_source"] == "schema.org")
    n_sch_parents = sum(len(r["children"]) for r in reparents if r["parent_source"] == "schema.org")
    # audit: did the embedder's shortlist actually propose each pair the LLM merged/re-parented?
    def _in_shortlist(a, b):
        return b in _adj.get(a, ()) or a in _adj.get(b, ())
    for _r in merges:
        for _m in _r["merged"]:
            _m["in_shortlist"] = _in_shortlist(_r["survivor_id"], _m["id"])
    for _r in reparents:
        for _c in _r["children"]:
            _c["in_shortlist"] = _in_shortlist(_r["parent_id"], _c["id"])
    for _r in generalizations:                    # no survivor id yet — audit against the first child
        _first = _r["merged"][0]["id"] if _r["merged"] else None
        for _m in _r["merged"]:
            _m["in_shortlist"] = _m["id"] == _first or _in_shortlist(_first, _m["id"])

    report = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "experiment": EXPERIMENT,
        "match_model": MATCH_MODEL,
        "gemini_model": GEMINI_MODEL, "expand_mode": EXPAND_MODE,
        "subprop_mode": SUBPROP_MODE, "source_vocab": str(VOCAB_PATH.name),
        "consolidated_vocab": str(CONSOLIDATED_VOCAB_PATH.name),
        "consolidate_top_k": CONSOLIDATE_TOP_K,
        "candidate_pool": ("vocab+schema.org" if CONSOLIDATE_INCLUDE_SCHEMA else "vocab"),
        "consolidation_reranked": RERANK_CONSOLIDATE,
        "schema_excluded_labels": sorted(SCHEMA_EXCLUDE_LABELS), "self_parent_fixed": self_parented,
        "neighbour_shortlist": shortlist, "neighbour_shortlist_reparent": shortlist_r,
        "articles_used": articles_before, "n_articles": len(articles_before),
        "vocab_size_before": len(ents), "vocab_size_after": len(vocab["properties"]),
        "allow_generalize": ALLOW_GENERALIZE, "guard_schema_parent": GUARD_SCHEMA_PARENT,
        "two_call_consolidation": TWO_CALL_CONSOLIDATION, "n_offkind_dropped": n_offkind,
        "n_merges": len(merges), "n_props_removed": len(merged_ids), "n_reparented": n_reparented,
        "n_merged_into_schema": n_into_schema, "n_reparented_under_schema": n_sch_parents,
        "n_generalize_groups": len(generalizations),
        "n_generalized": sum(len(r["merged"]) for r in generalizations),
        "n_blocked_degrounding": len(blocked_degrounding),
        "n_parent_conflicts": len(parent_conflicts), "n_keep_overrides": len(keep_overrides),
        "merges": merges, "reparents": reparents, "generalizations": generalizations,
        "blocked_degrounding": blocked_degrounding,
        "parent_conflicts": parent_conflicts, "keep_overrides": keep_overrides,
    }
    if write_report:
        CONSOLIDATION_REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False))

    if verbose:
        print(f"consolidation ({2 if TWO_CALL_CONSOLIDATION else 1} call): "
              f"subprop_mode={SUBPROP_MODE!r} | pool={report['candidate_pool']} | "
              f"{len(merges)} merge-group(s), {len(generalizations)} generalize-group(s), "
              f"{len(merged_ids)} removed ({n_into_schema} into schema.org), {n_reparented} re-parented "
              f"({n_sch_parents} under schema.org) | vocab {len(ents)} -> {len(vocab['properties'])}")
        for r in merges:
            tag = "schema.org" if r["survivor_source"] == "schema.org" else r["survivor_id"]
            print(f"  [merge/{r['kind'][:4]}] keep {tag} {r['survivor_label']!r} <= "
                  f"{[m['label'] for m in r['merged']]}")
        for r in generalizations:
            par = f" ⊑ {r['parent']}" if r["parent"] else ""
            print(f"  [generalize] new {r['new_id']} {r['new_label']!r}{par} <= "
                  f"{[m['label'] for m in r['merged']]}")
        for r in reparents:
            tag = " (schema.org)" if r["parent_source"] == "schema.org" else ""
            print(f"  [subproperty] {[c['label'] for c in r['children']]} subPropertyOf {r['parent_label']}{tag}")
        for k in keep_overrides:
            print(f"  [keep_id] survived as {k['kept']!r} (support {k['kept_support']}) over "
                  f"{k['most_supported']!r} (support {k['most_supported_support']})")
        for p in parent_conflicts:
            print(f"  [1-parent] {p['label']!r} kept under {p['kept_parent']}, "
                  f"refused second parent {p['rejected_parent']}")
        for b in blocked_degrounding:
            print(f"  [guard] kept {b['label']!r} under schema:{b['kept_parent']} "
                  f"(refused re-parent onto {b['rejected_parent']})")
        print(f"  consolidated vocab -> {CONSOLIDATED_VOCAB_PATH}  (raw {VOCAB_PATH.name} untouched)")
        if write_report:
            print(f"  report -> {CONSOLIDATION_REPORT_PATH}")
    return report

# Runs on the RAW incremental vocab and writes the consolidated result to a separate file.
# CONSOLIDATE=False (the `no-consolidation` ablation) skips the LLM but still writes both files, so
# the downstream stages behave identically and the metrics record a genuine zero, not a missing file.
_ = consolidate_vocab(load_vocab()) if CONSOLIDATE else _consolidation_disabled()

consolidation (2 call): subprop_mode='subproperty' | pool=vocab+schema.org | 6 merge-group(s), 0 generalize-group(s), 14 removed (0 into schema.org), 17 re-parented (17 under schema.org) | vocab 33 -> 19
  [merge/syno] keep P0022 'follows' <= ['followed', 'followedBy', 'precededBy', 'precedes', 'leadsTo', 'ledTo']
  [merge/syno] keep P0009 'affected' <= ['affects']
  [merge/syno] keep P0014 'announcedBy' <= ['introducedBy']
  [merge/syno] keep P0016 'involves' <= ['for', 'plans']
  [merge/syno] keep P0017 'launchVehicle' <= ['spacecraft', 'carriedBy', 'launches']
  [merge/syno] keep P0020 'tests' <= ['testFlightFor']
  [subproperty] ['popularIn'] subPropertyOf subjectOf (schema.org)
  [subproperty] ['approved'] subPropertyOf recognizingAuthority (schema.org)
  [subproperty] ['enables'] subPropertyOf potentialAction (schema.org)
  [subproperty] ['extendsTowards'] subPropertyOf geoCovers (schema.org)
  [subproperty] ['violates'] subPropertyOf contraindication (schema.org)
  [subproperty]

### Refresh `nary_graph_extended.json` after consolidation → **separate** files — no LLM

Apply the consolidation's `merged_id → survivor` mapping (and subproperty re-parentings) to the
per-article files deterministically, writing the result to a **new** `nary_graph_extended_consolidated.json`
per article — the incremental `nary_graph_extended.json` is left untouched. When a property merged
**into schema.org**, the record stops being an extension property: `property_source` flips to
`"schema.org"` and `vocab_id` / `property_definition` / `property_parent` go null, exactly like a
record the build had labelled with schema.org directly. Also emit one global
`relation_consolidation_ext_changes.json` listing **only** the records that changed (before → after).

In [22]:
# Produce a CONSOLIDATED copy of each per-article nary_graph_extended.json WITHOUT the LLM — the incremental
# nary_graph_extended.json files are left UNTOUCHED. Three effects, applied only to the copy:
#   - MERGED properties: records pointing at a removed vocab_id are remapped to its survivor.
#   - GENERALIZED properties: records pointing at a specific that was folded into a newly minted
#     general property are remapped onto that new property.
#   - MERGED INTO SCHEMA.ORG: when the survivor is a schema.org property the record stops being an
#     extension property — property_source becomes "schema.org", vocab_id/definition/parent null.
#   - RE-PARENTED properties (subproperty mode): the property survives, so we re-sync every vocab-typed
#     record's label / definition / parent from the CONSOLIDATED vocabulary (picks up the new parent).
# `property_is_new_to_vocab` is a BUILD-TIME provenance fact — consolidation leaves it untouched.
# Writes, per article, nary_graph_extended_consolidated.json, and ONE global changes file listing ONLY the
# records that changed (before -> after). Pure bookkeeping. Run right after consolidate_vocab.
EXT_CONSOLIDATED_NAME = f"nary_graph{_osfx}_extended_consolidated.json"                       # per-article refreshed copy
EXT_CHANGES_PATH      = RESULTS_DIR / f"_global/e2e/nary_relation_consolidation_ext_changes{_osfx}.json" # global "what changed" file
_CHANGED_FIELDS = ("vocab_id", "property_source", "property", "property_definition", "property_parent")  # substantive fields

def refresh_ext_after_consolidation(report=None, verbose=True):
    if report is None:
        if not CONSOLIDATION_REPORT_PATH.exists():
            print("no consolidation report found — run consolidate_vocab first"); return
        report = json.loads(CONSOLIDATION_REPORT_PATH.read_text())
    # merged vocab id -> where it went: another vocab id, or a schema.org property label
    remap = {}
    for r in report.get("merges", []):
        tgt = ({"source": "schema.org", "label": r["survivor_label"]}
               if r.get("survivor_source") == "schema.org" else
               {"source": "vocab", "id": r["survivor_id"]})
        for m in r.get("merged", []):
            remap[m["id"]] = tgt
    # GENERALIZED properties: the survivor is a property MINTED during consolidation, so its id
    # appears nowhere in the incremental files — every specific it replaced has to be remapped onto
    # it here, or those records keep pointing at an id the consolidated vocabulary no longer has.
    for r in report.get("generalizations", []):
        for m in r.get("merged", []):
            remap[m["id"]] = {"source": "vocab", "id": r["new_id"]}
    by_id = {t["id"]: t for t in load_consolidated()["properties"]}   # survivors + re-parented properties

    def _resolve(v):
        """The consolidated fields for a record currently using vocab id `v` (None: leave as-is).
        Follows merge chains (a survivor of one group can be merged again by a later one)."""
        tgt, seen = remap.get(v), set()
        while tgt is not None and tgt["source"] == "vocab" and tgt["id"] in remap and tgt["id"] not in seen:
            seen.add(tgt["id"]); tgt = remap[tgt["id"]]
        if tgt is not None and tgt["source"] == "schema.org":         # left the extension for schema.org
            return {"vocab_id": None, "property_source": "schema.org", "property": tgt["label"],
                    "property_definition": None, "property_parent": None}
        s = by_id.get(tgt["id"] if tgt else v)                        # follow a merge, else same id
        if s is None:
            return None
        return {"vocab_id": s["id"], "property_source": "vocab", "property": s["canonical_label"],
                "property_definition": s.get("definition"), "property_parent": s.get("parent")}

    all_changes, total = [], 0
    for fp in sorted(RESULTS_DIR.glob(f"*/e2e/nary_graph{_osfx}_extended.json")):
        data = json.loads(fp.read_text())                            # the untouched incremental file
        aid  = data.get("article_id", fp.parent.name)
        art_changes = []
        for n in data["relations"]:
            v = n.get("vocab_id")
            if not v:
                continue
            new = _resolve(v)
            if new is None:
                continue
            old = {k: n.get(k) for k in _CHANGED_FIELDS}
            if new != old:
                n.update(new)                                        # mutate the in-memory COPY only
                art_changes.append({"subject_text": n.get("subject_text"), "object_text": n.get("object_text"),
                                    "before": old, "after": new})
        data["consolidated"], data["subprop_mode"] = True, SUBPROP_MODE
        (fp.parent / EXT_CONSOLIDATED_NAME).write_text(json.dumps(data, indent=2, ensure_ascii=False))
        if art_changes:
            all_changes.append({"article_id": aid, "n_changed": len(art_changes), "changes": art_changes})
            total += len(art_changes)
            if verbose:
                print(f"  {aid}: {len(art_changes)} record(s) changed")

    EXT_CHANGES_PATH.write_text(json.dumps({
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "subprop_mode": SUBPROP_MODE,
        "consolidated_vocab": str(CONSOLIDATED_VOCAB_PATH.name),
        "n_articles_changed": len(all_changes), "n_records_changed": total, "articles": all_changes,
    }, indent=2, ensure_ascii=False))
    print(f"done — wrote {EXT_CONSOLIDATED_NAME} for each article (incremental nary_graph_extended.json untouched)\n"
          f"       {total} changed record(s) across {len(all_changes)} article(s) -> {EXT_CHANGES_PATH.name}")

refresh_ext_after_consolidation()

  art10: 265 record(s) changed
  art13: 6 record(s) changed
  art14: 4 record(s) changed
  art16: 2 record(s) changed
  art6: 2 record(s) changed
  art7: 4 record(s) changed
  art9: 12 record(s) changed
done — wrote nary_graph_dedup_main_test_extended_consolidated.json for each article (incremental nary_graph_extended.json untouched)
       295 changed record(s) across 7 article(s) -> nary_relation_consolidation_ext_changes_dedup_main_test.json


In [23]:
# ---- PERIODIC RE-CONSOLIDATION (CONSOLIDATE_EVERY) -------------------------------------------
# The relation-side twin of 6d's periodic cell; see it for why the passes are accumulated instead of
# refreshed one at a time (refresh_ext_after_consolidation composes merge chains from ONE report, so
# refreshing per pass would drop the earlier ones and strand article 1 on a retired property id).
#
# Two things kept from the main build loop, which 6d does not have:
#   prefetch   within a chunk the per-article calls are still independent while the inline pool is
#              schema.org only, so PARALLEL_LLM keeps working — it is only ACROSS chunks that the
#              vocabulary state matters.
#   tokens     TOKENS accumulates across chunks and the ledger is rewritten at the end, so the cost
#              file covers the whole stage rather than the first chunk.
if CONSOLIDATE_EVERY and len(TODO_ALL) > len(todo):
    _reports = [json.loads(CONSOLIDATION_REPORT_PATH.read_text())] if CONSOLIDATION_REPORT_PATH.exists() else []
    _rest = TODO_ALL[len(todo):]
    _chunks = [_rest[i:i + CONSOLIDATE_EVERY] for i in range(0, len(_rest), CONSOLIDATE_EVERY)]
    print(f"periodic consolidation: {len(_chunks)} further chunk(s) of <= {CONSOLIDATE_EVERY} article(s)\n")

    for _ci, _chunk in enumerate(_chunks, start=2):
        # RECONCILING feed-back. A pass must only ever remove a property it RECORDED as merged;
        # anything else that disappears is a term the per-article graphs still point at, and feeding
        # a lossy vocabulary forward turns those references into dangling ids that no later pass can
        # repair (it no longer knows the term existed). Measured on the first periodic 6e run: 5 of
        # 29 removals had no merge record, orphaning 9 relations. Restore them and say so, rather
        # than carrying the loss silently into the next chunk.
        _prev = {t["id"]: t for t in load_vocab()["properties"]}
        _con = load_consolidated()
        _merged = {m["id"] for g in (_reports[-1].get("merges") or []) for m in (g.get("merged") or [])}
        _lost = [i for i in _prev if i not in {t["id"] for t in _con["properties"]} and i not in _merged]
        if _lost:
            print(f"  [repair] {len(_lost)} property(s) left the vocabulary with no merge record "
                  f"-- restoring: {_lost}")
            _con["properties"] = _con["properties"] + [_prev[i] for i in _lost]
        save_vocab(_con)
        vocab = load_vocab()
        print(f"── chunk {_ci}: {len(vocab['properties'])} propert(ies) carried forward | {_chunk}")
        _pre = {}
        if PARALLEL_LLM and not INLINE_VOCAB_CANDIDATES and len(_chunk) > 1:
            _pre = {r.item: r.value for r in
                    nb_parallel.pmap(lambda a: prefetch_assignments(a, vocab), _chunk, workers=WORKERS)
                    if r.ok}
        for aid in _chunk:
            _ta = time.perf_counter()
            with nb_parallel.usage_scope(aid):
                process_article(aid, vocab, write=True, save=True, verbose=True, pre=_pre.get(aid))
            _u = (_pre.get(aid) or {}).get("usage") or LAST_USAGE
            TOKENS.append({"article": aid, "seconds": round(time.perf_counter() - _ta, 1),
                           **(_u or {})})
            if aid not in _pre:
                time.sleep(5)
        if CONSOLIDATE:
            _reports.append(consolidate_vocab(load_vocab()))
        else:
            _consolidation_disabled()

    if _reports:
        _cum = dict(_reports[-1])
        _cum["merges"] = [g for r in _reports for g in (r.get("merges") or [])]
        _cum["reparents"] = [g for r in _reports for g in (r.get("reparents") or [])]
        _cum["consolidation_passes"] = len(_reports)
        _cum["n_merges"] = len(_cum["merges"])
        _cum["n_reparented"] = sum(len(g.get("children") or []) for g in _cum["reparents"])
        _cum["n_props_removed"] = sum(len(g.get("merged") or []) for g in _cum["merges"])
        _cum["n_merged_into_schema"] = sum(len(g.get("merged") or []) for g in _cum["merges"]
                                           if g.get("survivor_source") == "schema.org")
        # NOT pass 1's "before" and pass 4's "after" — those describe different populations (the
        # vocabulary after one chunk vs after twenty), and quoted as a pair they read as though
        # consolidation GREW the vocabulary. The meaningful pair for a periodic run is the
        # counterfactual: what the vocabulary would be had nothing been collapsed, against what it
        # actually is. (Sanity check: before == next_id - 1, i.e. every id ever issued.)
        _cum["vocab_size_after"] = _reports[-1].get("vocab_size_after")
        _cum["vocab_size_before"] = (_cum["vocab_size_after"] or 0) + _cum["n_props_removed"]
        _cum["per_pass"] = [{"before": r.get("vocab_size_before"), "after": r.get("vocab_size_after")}
                            for r in _reports]
        CONSOLIDATION_REPORT_PATH.write_text(json.dumps(_cum, indent=2, ensure_ascii=False))
        print(f"\ncumulative report over {len(_reports)} pass(es) -> {CONSOLIDATION_REPORT_PATH.name}"
              f"   {_cum['n_merges']} merge group(s), {_cum['n_props_removed']} propert(ies) removed")

    refresh_ext_after_consolidation()          # once, with the composed mapping

    _toktot = {k: sum((t.get(k) or 0) for t in TOKENS)
               for k in ("prompt_tokens", "output_tokens", "thoughts_tokens", "total_tokens", "seconds")}
    _tokp = RESULTS_DIR / "_global" / "e2e" / f"token_usage{_osfx}.json"
    _tokp.write_text(json.dumps({"method": "candidate", "experiment": EXPERIMENT, "variant": VARIANT,
                                 "prompt": PROMPT, "model": GEMINI_MODEL,
                                 "consolidation_passes": len(_reports),
                                 "per_article": TOKENS, "totals": _toktot}, indent=2, ensure_ascii=False))
    print(f"\ndone — {len(load_consolidated()['properties'])} consolidated properties over "
          f"{len(TODO_ALL)} article(s) | tokens {_toktot['total_tokens']:,} -> {_tokp.name}")

# ---- INVARIANT: the consolidated graphs and the vocabulary must agree ---------------------------
# Consolidating repeatedly can strand a graph on an id that no longer exists: a term merged in one
# pass and merged AGAIN in a later one loses its trail. That failure is SILENT — the run finishes and
# the counts look plausible. Three cheap checks, each of which has caught a real defect on this
# corpus:
#   dangling    an id the vocabulary no longer holds (the original check).
#   duplicates  the SAME id held twice. A repair run that reused a run_tag re-issued P0558-P0563,
#               already taken by six consolidation-generalize terms, and six relations then resolved
#               to a different property than the one they were assigned. Ids still existed, so the
#               dangling check saw nothing.
#   labels      the reference's own label vs the label of the entry it points at. This is what
#               catches a refresh that did not run: nb4's read the wrong field names, remapped
#               nothing, and reported "0 nodes changed" while 65 references stayed on
#               pre-consolidation terms.
import glob as _glob
_vocab = load_consolidated()["properties"]
_live = {t["id"]: t.get("canonical_label") for t in _vocab}
_dups = len(_vocab) - len(_live)
_dangling, _mismatch, _checked = [], [], 0
for _f in sorted(_glob.glob(str(RESULTS_DIR / "art*" / "e2e" / f"*{_osfx}_extended_consolidated.json"))):
    _g = json.loads(Path(_f).read_text())
    _aid = Path(_f).parts[-3]
    for _coll in ("relations",):
        for _n in _g.get(_coll, []):
            _v = _n.get("vocab_id")
            if not _v:
                continue
            _checked += 1
            _lab = _n.get("property")
            if _v not in _live:
                _dangling.append((_aid, _v, _lab))
            elif _lab != _live[_v]:
                _mismatch.append((_aid, _v, _lab, _live[_v]))
if _dups:
    from collections import Counter as _C
    _seen = _C(t["id"] for t in _vocab)
    print(f"\n  *** {_dups} duplicated id(s) in the consolidated vocabulary: "
          f"{[i for i, n in _seen.items() if n > 1][:10]}")
if _dangling:
    print(f"\n  *** {len(_dangling)} reference(s) to {len({d[1] for d in _dangling})} vanished property(s):")
    for _d in _dangling[:10]:
        print(f"      {_d[0]}  {_d[1]}  {_d[2]}")
if _mismatch:
    print(f"\n  *** {len(_mismatch)} reference(s) whose label disagrees with the vocabulary "
          f"(a refresh that did not reach this data):")
    for _d in _mismatch[:10]:
        print(f"      {_d[0]}  {_d[1]}  graph says {_d[2]!r}, vocabulary says {_d[3]!r}")
if _dups or _dangling or _mismatch:
    raise AssertionError(
        f"{_dups} duplicated id(s), {len(_dangling)} dangling reference(s) and "
        f"{len(_mismatch)} label disagreement(s) between the consolidated graphs and the "
        f"property vocabulary — these graphs are NOT safe to report. Fix before using this run.")
print(f"  invariant OK — {_checked} vocabulary reference(s) resolve, with matching labels and no "
      f"duplicated ids")


periodic consolidation: 14 further chunk(s) of <= 5 article(s)

── chunk 2: 19 propert(ies) carried forward | ['art13', 'art14', 'art16', 'art20', 'art21']
5 item(s), running on 5 threads

[1/5] art20  (50.6s)
[2/5] art21  (73.1s)
[3/5] art13  (73.9s)
[4/5] art14  (80.4s)
[5/5] art16  (87.5s)

done — 5/5 ok in 87.5s
  art13: 53 phrases (0 split), 164 instances -> 164 triples (0 merged, 0 inverse-reused) | +3 minted | vocab now 22
  art14: 41 phrases (0 split), 59 instances -> 59 triples (0 merged, 2 inverse-reused) | +23 minted | vocab now 45
  art16: 33 phrases (0 split), 889 instances -> 889 triples (0 merged, 10 inverse-reused) | +1 minted | vocab now 46
  art20: 20 phrases (0 split), 48 instances -> 48 triples (0 merged, 0 inverse-reused) | +1 minted | vocab now 47
  art21: 34 phrases (0 split), 68 instances -> 68 triples (0 merged, 2 inverse-reused) | +0 minted | vocab now 47
consolidation (2 call): subprop_mode='subproperty' | pool=vocab+schema.org | 24 merge-group(s), 0 generali

## Property hierarchy (schema.org + extension) — raw and consolidated, **separate** files

One indented tree of the property space — schema.org properties (linked by `subPropertyOf`) and the
extension (minted) properties together, each tagged `[schema.org]` or `[extension]`, sub-properties
nested under their parent. Built **twice**: `property_hierarchy_ext.txt` from the **raw incremental**
vocab, and `property_hierarchy_ext_consolidated.txt` from the **consolidated** vocab. Set
`INCLUDE_ALL_SCHEMA=True` to include every schema.org property, or leave it `False` to show just the
branches your extension properties hang off (extension properties + their super-property chains).

In [24]:
# Property hierarchy: schema.org properties + extension (vocab) properties in ONE indented tree, each
# tagged [schema.org] or [extension]; sub-properties nested under their subPropertyOf parent.
#   property_hierarchy_ext.txt              <- raw incremental vocab (relation_vocabulary.json)
#   property_hierarchy_ext_consolidated.txt <- consolidated vocab   (re-parentings/merges reflected)
# INCLUDE_ALL_SCHEMA comes from the experiment config:
#   False -> only extension properties + their super-property chains.
#   True  -> the entire schema.org property set too (large).
HIERARCHY_PATH                   = RESULTS_DIR / f"_global/e2e/property_hierarchy_ext{_osfx}.txt"                # raw incremental
HIERARCHY_CONSOLIDATED_PATH      = RESULTS_DIR / f"_global/e2e/property_hierarchy_ext{_osfx}_consolidated.txt"   # consolidated

def build_property_hierarchy(vocab, write_path, include_all_schema=INCLUDE_ALL_SCHEMA, show=50):
    schema_labels = {p["label"] for p in schema_pool}
    schema_parent = {p["label"]: _schema_super(p["label"]) for p in schema_pool}
    ext = {t["canonical_label"]: t for t in vocab["properties"]}

    parent_of, kind = {}, {}
    for lbl in schema_labels:
        parent_of[lbl] = schema_parent.get(lbl); kind[lbl] = "schema.org"
    for lbl, t in ext.items():                                   # extension overrides if same label
        parent_of[lbl] = t.get("parent"); kind[lbl] = "extension"

    if include_all_schema:
        included = set(schema_labels) | set(ext)
    else:                                                        # only extensions + their ancestor chains
        included = set(ext)
        for lbl in list(ext):
            cur, seen = parent_of.get(lbl), set()
            while cur and cur not in seen and cur not in included:
                seen.add(cur); included.add(cur); cur = parent_of.get(cur)

    children, roots = defaultdict(list), []
    for lbl in included:
        p = parent_of.get(lbl)
        (children[p].append(lbl) if p in included else roots.append(lbl))
    for p in children:
        children[p].sort()

    lines = []
    def walk(lbl, depth, seen):
        tag   = "[extension]" if kind.get(lbl) == "extension" else "[schema.org]"
        extra = f"  (support {ext[lbl].get('support', 0)})" if lbl in ext else ""
        lines.append("  " * depth + f"{lbl}  {tag}{extra}")
        for c in children.get(lbl, []):
            if c not in seen:
                walk(c, depth + 1, seen | {c})
    for r in sorted(set(roots)):
        walk(r, 0, {r})

    text = "\n".join(lines)
    if write_path:
        write_path.write_text(text)
    print(f"property hierarchy: {len(lines)} nodes ({len(ext)} extension) | "
          f"include_all_schema={include_all_schema} -> {write_path}\n" + "-" * 74)
    print(text if len(lines) <= show else "\n".join(lines[:show]) + f"\n  ... ({len(lines) - show} more — see file)")
    return text

# Raw incremental hierarchy (always available), then the consolidated one (reflects re-parentings/merges).
print("=== RAW incremental property vocabulary ===")
build_property_hierarchy(load_vocab(),        HIERARCHY_PATH)
print("\n=== CONSOLIDATED property vocabulary ===")
build_property_hierarchy(load_consolidated(), HIERARCHY_CONSOLIDATED_PATH)

=== RAW incremental property vocabulary ===
property hierarchy: 120 nodes (127 extension) | include_all_schema=False -> /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/deepseek-v4-flash/_global/e2e/property_hierarchy_ext_dedup_main_test.txt
--------------------------------------------------------------------------
advised  [extension]  (support 1)
affects  [extension]  (support 20)
affectsRiver  [extension]  (support 4)
after  [extension]  (support 1)
allowsCharge  [extension]  (support 1)
announced  [extension]  (support 1)
announcedBy  [extension]  (support 2)
asserts  [extension]  (support 3)
  administeredDuring  [extension]  (support 1)
  affected  [extension]  (support 2)
    injured  [extension]  (support 1)
  appearsRedDueTo  [extension]  (support 2)
  appliesTo  [extension]  (support 1)
  datesFrom  [extension]  (support 3)
  determinedPowerSourceOf  [extension]  (support 1)
  discovered  [extension]  (support 1)
    discoveredBy  [ext

'advised  [extension]  (support 1)\n  consults  [extension]  (support 1)\naffects  [extension]  (support 24)\n  affected  [extension]  (support 5)\nallowsCharge  [extension]  (support 1)\nasserts  [extension]  (support 3)\n  administeredDuring  [extension]  (support 1)\n  appearsRedDueTo  [extension]  (support 2)\n  appliesTo  [extension]  (support 1)\n  datesFrom  [extension]  (support 3)\n  determinedPowerSourceOf  [extension]  (support 1)\n  discovered  [extension]  (support 1)\n    discoveredBy  [extension]  (support 1)\n  hadPassenger  [extension]  (support 11)\n  hasRate  [extension]  (support 23)\n  indicates  [extension]  (support 1)\n  ledTo  [extension]  (support 4)\n    precededBy  [extension]  (support 3)\n  locatedAt  [extension]  (support 1)\n  notDueTo  [extension]  (support 1)\n  occurredOn  [extension]  (support 1)\n    occursIn  [extension]  (support 1)\n  outshines  [extension]  (support 2)\n  poweredBy  [extension]  (support 2)\n    notPoweredBy  [extension]  (suppo

## Run metrics — every number this stage produced, in one file

`run_metrics.relations` re-reads the review files, the extended graphs, the grown vocabulary and the
consolidation report, and reports what the write-up needs: distinct relation phrases, how many
**split** across properties, instances vs. deduplicated triples, **inverse reuse**, the
schema.org-vs-mint split per assignment and per stored triple, the **retrieval quality** that split
depends on (was the chosen property in the shortlist at all, and at which rank), the vocabulary size
**before and after** consolidation, what consolidation did (merges, generalizations, folds into
schema.org, re-parentings, blocked de-groundings, parent conflicts, keep-id overrides, and the
`in_shortlist` audit), and the token cost.

Saved to `_global/e2e/metrics_relations<_osfx>.json`. `run_metrics.compare(...)` lines several
experiments up afterwards — which is the table the retrieval/consolidation ablations exist to fill.

In [25]:
# Collect + persist this stage's numbers. Pure reading — no LLM call, safe to re-run, and it works
# on a run that finished days ago.
import importlib, run_metrics
importlib.reload(run_metrics)

_m = run_metrics.relations(RESULTS_DIR, ARTICLE_ORDER, isfx=_isfx, osfx=_osfx,
                           track="e2e", experiment=EXPERIMENT)
run_metrics.show(_m)

# save() also CLOSES this stage in the run manifest — the record of WHICH run these numbers
# came from: the resolved config, the git commit, the article order, every output file and
# the token spend.  results/<model>/_runs/<run_id>.json  ·  python -m run_manifest list
print("\nsaved ->", run_metrics.save(_m, RESULTS_DIR, _osfx, "e2e",
                                     section="6e", articles=ARTICLE_ORDER))

# Side by side with other experiments already run and saved (missing ones are skipped):
# print(run_metrics.compare(RESULTS_DIR, "relations",
#                           ["_dedup_main", "_dedup_no_rerank", "_dedup_concise"], "e2e"))

RELATIONS  ·  e2e  ·  experiment 'deepseek-loop-drop'
article          phrases          mints  split_phrases      instances        triples  merged_triple  inverse_reuse  schema_org_tr  vocab_triples  triples_conso  schema_org_tr  cons_property  cons_triples_     cum_minted
art6                  32              4              0             52             52              0             11             48              4             52             52              4              4              4
art7                  38              4              0             70             70              0              5             66              4             70             70              4              4              8
art9                  27              5              1             84             84              0             17             72             12             84             84             12             12             13
art10                 36             20              0            